# Machine Learning Techniques: A Practical Guide

This notebook provides hands-on exercises covering key machine learning preprocessing and modeling techniques:
1. Data Imputation
2. Feature Engineering (Numeric & Categorical Transformations)
3. Regularization
4. Feature Selection
5. Classes Rebalancing
6. Ensemble Methods
7. Probability Calibration
8. Model Depreciation (Drift Detection)
9. Model Evaluation & Diagnostics
10. Cross-Validation Variants
11. Bayesian Hyperparameter Search

Author: Michał Woźniak

## Technical Setup: Running This Notebook Locally with `uv`

### Prerequisites
- Ensure you have `uv` installed on your system. If not, install it following the instructions at [https://github.com/astral-sh/uv](https://github.com/astral-sh/uv)

### Step 1: Initialize a New Project
Navigate to your working directory and initialize a new Python project:
```bash
# Navigate to your project directory
cd /path/to/your/project

# Initialize uv project (if not already initialized)
uv init

# Or if you want to specify Python version
uv init --python 3.11
```

### Step 2: Install Required Dependencies
Install all necessary packages for this notebook:
```bash
# Install core data science packages
uv add numpy pandas scikit-learn scipy

# Install visualization libraries
uv add matplotlib seaborn

# Install imbalanced-learn (for resampling techniques)
uv add imbalanced-learn

# Install Jupyter notebook
uv add jupyter ipykernel

# Alternative: Install all at once
uv add numpy pandas scikit-learn scipy matplotlib seaborn imbalanced-learn jupyter ipykernel
```

### Step 3: Activate the Environment and Start Jupyter
```bash
# Activate the virtual environment (uv handles this automatically)
# Start Jupyter notebook
uv run jupyter notebook

# Or use Jupyter Lab
uv run jupyter lab
```

### Step 4: Open This Notebook
Once Jupyter starts, navigate to `MLTechniques.ipynb` and open it.

### Required Packages Summary
- **numpy** (≥1.21.0): Numerical computations and array operations
- **pandas** (≥1.3.0): Data manipulation and analysis
- **scikit-learn** (≥1.0.0): Machine learning algorithms and utilities
- **scipy** (≥1.7.0): Scientific computing (statistical tests, distance metrics)
- **imbalanced-learn** (≥0.12.0): Resampling techniques for imbalanced datasets (SMOTE, Tomek Links, etc.)
- **matplotlib** (≥3.4.0): Basic plotting and visualization
- **seaborn** (≥0.11.0): Statistical data visualization
- **jupyter**: Jupyter notebook interface
- **ipykernel**: Jupyter kernel for Python

### Troubleshooting
- If you encounter import errors, ensure all packages are installed: `uv pip list`
- To update packages: `uv add --upgrade package_name`
- To see your Python version: `uv run python --version`

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

np.random.seed(42)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Libraries imported successfully!")

# 1. Data Imputation

When working with real data you will always encounter the problem of **missing values**. Causes include: data does not exist, hardware/software/human error, data was deleted, etc.

## Types of Missing Data

| Type | Name | Description |
|------|------|-------------|
| **MCAR** | Missing Completely At Random | Missingness is unrelated to observed and unobserved features |
| **MAR** | Missing At Random | Missingness is related to observed features only |
| **NMAR** | Not Missing At Random | Missingness is related to unobserved features (and possibly observed features) |

## Techniques for dealing with missing values:
- **Do nothing**: some ML algorithms handle missing values automatically
- **Remove columns**: when variable has >10% missing and is not crucial
- **Remove rows**: avoid if possible, especially with small datasets
- **Imputation**: fill in missing values using univariate or multivariate techniques

## 1.1 Creating a Dataset with Missing Values

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df_full_housing = housing.frame.copy()
X_full = housing.data.copy()
y_full = housing.target.copy()

# Subsample for imputation exercises (KNN/MICE are expensive on large datasets)
df = df_full_housing.sample(n=2000, random_state=42).reset_index(drop=True)

# Introduce missing values of different types
rng = np.random.default_rng(42)
df_missing = df.copy()

# MCAR: randomly remove 5% of MedInc values
# The probability of MedInc being missing is pure random chance — it has nothing to do with the value of MedInc itself, nor any other column. Every row has exactly 5% chance of losing its value, regardless of whether it's a rich or poor neighborhood.
mcar_mask = rng.random(len(df)) < 0.05
df_missing.loc[mcar_mask, "MedInc"] = np.nan
# Real-world analogy: a sensor randomly fails, or a survey respondent randomly skips a question by accident.

# MAR: remove AveRooms when Population > 2000
# AveRooms goes missing depending on another observed column (Population), not on AveRooms itself. High-population areas are more likely to have AveRooms missing — but once you know Population, the missingness is explainable and random within that group.
mar_mask = (df_missing["Population"] > 2000) & (rng.random(len(df)) < 0.3)
df_missing.loc[mar_mask, "AveRooms"] = np.nan
# Real-world analogy: lower-income respondents are less likely to report their salary — but income bracket (observable) explains why.

# NMAR: remove HouseAge for older houses
# HouseAge goes missing because of its own value — old houses (age > 40) are more likely to have their age missing. The missingness is driven by the variable that's missing, which you can no longer observe.
nmar_mask = (df_missing["HouseAge"] > 40) & (rng.random(len(df)) < 0.4)
df_missing.loc[nmar_mask, "HouseAge"] = np.nan
# Real-world analogy: very sick patients skipping health questionnaires — the worst outcomes are systematically underreported.

print("Missing values summary:")
missing = df_missing.isnull().sum()
pct = (100 * missing / len(df_missing)).round(2)
print(pd.DataFrame({"Count": missing, "Percent": pct})[missing > 0])

## 1.2 Visualizing Missing Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sample = df_missing.sample(200, random_state=42).sort_index()
sns.heatmap(sample.isnull(), cbar=True, yticklabels=False, cmap="viridis", ax=axes[0])
axes[0].set_title("Missing Data Heatmap (sample of 200 rows)")

missing_pct = df_missing.isnull().sum() / len(df_missing) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
missing_pct.plot(kind="bar", ax=axes[1], color="coral", edgecolor="black")
axes[1].set_title("Percentage of Missing Values by Feature")
axes[1].set_ylabel("Missing (%)")

plt.tight_layout()
plt.show()

## 1.3 Removing Missing Data

In [ ]:
print("Original shape:", df_missing.shape)

# Drop columns with > 10% missing
threshold = 0.10
cols_to_drop = df_missing.columns[df_missing.isnull().mean() > threshold]
print(f"Columns with >{threshold * 100:.0f}% missing: {list(cols_to_drop)}")

df_dropped_cols = df_missing.drop(columns=cols_to_drop)
print(f"Shape after dropping columns: {df_dropped_cols.shape}")

# Drop rows with any missing value
df_dropped_rows = df_missing.dropna()
print(f"Shape after dropping rows: {df_dropped_rows.shape}")
print(
    f"Rows lost: {len(df_missing) - len(df_dropped_rows)} ({(1 - len(df_dropped_rows) / len(df_missing)) * 100:.1f}%)"
)

## 1.4 Univariate Imputation (Single Feature)

Uses information from only one column at a time.

- **Continuous variables**: constant (e.g. 0), mean, median, mode, random value from distribution
- **Categorical variables**: new "missing" category, mode, random replacement
- **Time series**: last/next observed value, linear/polynomial/spline interpolation

In [ ]:
from sklearn.impute import SimpleImputer

# Constant imputation
imp_constant = SimpleImputer(strategy="constant", fill_value=0)
df_const = df_missing.copy()
df_const["MedInc"] = imp_constant.fit_transform(df_missing[["MedInc"]])

# Mean imputation
imp_mean = SimpleImputer(strategy="mean")
df_mean = df_missing.copy()
df_mean["MedInc"] = imp_mean.fit_transform(df_missing[["MedInc"]])

# Median imputation
imp_median = SimpleImputer(strategy="median")
df_median = df_missing.copy()
df_median["MedInc"] = imp_median.fit_transform(df_missing[["MedInc"]])

# Most frequent (mode) imputation
imp_mode = SimpleImputer(strategy="most_frequent")
df_mode = df_missing.copy()
df_mode["MedInc"] = imp_mode.fit_transform(df_missing[["MedInc"]])

print("SimpleImputer strategies: constant, mean, median, most_frequent")
print(f"  Mean fill:   {imp_mean.statistics_[0]:.4f}")
print(f"  Median fill: {imp_median.statistics_[0]:.4f}")

In [ ]:
# Visualize the effect of different univariate strategies
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)

strategies = [("Constant (0)", df_const), ("Mean", df_mean), ("Median", df_median), ("Mode", df_mode)]
for ax, (name, frame) in zip(axes, strategies):
    ax.hist(df["MedInc"], bins=50, alpha=0.5, label="Original", density=True)
    ax.hist(frame["MedInc"], bins=50, alpha=0.5, label=f"Imputed ({name})", density=True)
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.set_xlabel("MedInc")

axes[0].set_ylabel("Density")
plt.suptitle("Effect of Univariate Imputation on MedInc Distribution", y=1.02)
plt.tight_layout()
plt.show()

### Missing Indicator

Sometimes the fact that a value was missing carries information — add a binary indicator.

In [ ]:
from sklearn.impute import MissingIndicator

indicator = MissingIndicator(features="missing-only")
missing_flags = indicator.fit_transform(df_missing)
print(f"Features with indicators: {df_missing.columns[indicator.features_].tolist()}")

# Combine imputed data with missing indicators
imp = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(imp.fit_transform(df_missing), columns=df_missing.columns)
for i, col in enumerate(df_missing.columns[indicator.features_]):
    df_imputed[f"{col}_was_missing"] = missing_flags[:, i].astype(int)

print(f"Shape with indicators: {df_imputed.shape}")
print(df_imputed.head())

## 1.5 Multivariate Imputation (Multiple Features)

Uses information from all other features to estimate missing values.

- **KNN Imputer**: uses K-nearest neighbors to impute
- **Iterative Imputer (MICE)**: Multivariate Imputation by Chained Equations

**MICE** 

The core idea: instead of filling each missing value independently (like median or KNN), train a regression model to predict each missing column using all other columns.

**How the algorithm runs**

Given a dataset with missing values in columns A, B, C:

Round 0 — initialization: fill all missing values with a simple estimate (e.g. mean) as a starting point.

Each iteration (repeated max_iter=10 times):

1. For column A: treat it as the target, use B and C as features → train the estimator → predict missing A values
2. For column B: treat it as the target, use A and C as features → predict missing B values
3. For column C: same...
4. Repeat for all columns with missing data

Each pass refines previous estimates because updated imputations from step 1 feed into step 2, etc. After 10 iterations, values converge.

In [ ]:
from sklearn.impute import KNNImputer

for k in [3, 5, 10]:
    knn_imp = KNNImputer(n_neighbors=k, weights="uniform")
    df_knn = pd.DataFrame(knn_imp.fit_transform(df_missing), columns=df_missing.columns)
    mae = np.abs(df["MedInc"][mcar_mask] - df_knn["MedInc"][mcar_mask]).mean()
    print(f"KNN (k={k:>2d}, uniform):  MAE on MedInc MCAR = {mae:.4f}")

knn_imp_dist = KNNImputer(n_neighbors=5, weights="distance")
df_knn_dist = pd.DataFrame(knn_imp_dist.fit_transform(df_missing), columns=df_missing.columns)
mae = np.abs(df["MedInc"][mcar_mask] - df_knn_dist["MedInc"][mcar_mask]).mean()
print(f"KNN (k= 5, distance): MAE on MedInc MCAR = {mae:.4f}")

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor

estimators = {
    "LinearRegression": LinearRegression(),
    "KNN (k=5)": KNeighborsRegressor(n_neighbors=5, weights="uniform"),
}

for name, estimator in estimators.items():
    mice_imp = IterativeImputer(estimator=estimator, max_iter=10, random_state=42)
    df_mice = pd.DataFrame(mice_imp.fit_transform(df_missing), columns=df_missing.columns)
    mae = np.abs(df["MedInc"][mcar_mask] - df_mice["MedInc"][mcar_mask]).mean()
    print(f"MICE ({name:>15s}): MAE on MedInc MCAR = {mae:.4f}")

In [ ]:
# Visual comparison: Original vs KNN vs MICE
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

knn5 = KNNImputer(n_neighbors=5, weights="distance")
df_knn5 = pd.DataFrame(knn5.fit_transform(df_missing), columns=df_missing.columns)

mice = IterativeImputer(estimator=LinearRegression(), max_iter=10, random_state=42)
df_mice_final = pd.DataFrame(mice.fit_transform(df_missing), columns=df_missing.columns)

for ax, (name, frame) in zip(
    axes, [("Original", df), ("KNN (k=5)", df_knn5), ("MICE (LinearRegression)", df_mice_final)]
):
    ax.hist(frame["MedInc"], bins=50, alpha=0.7, edgecolor="black")
    ax.set_title(name)
    ax.set_xlabel("MedInc")
    ax.set_ylabel("Count")

plt.suptitle("Distribution Comparison: Original vs Imputed")
plt.tight_layout()
plt.show()

## 1.6 Imputation in a Pipeline (Preventing Leakage!)

**Critical**: imputation must be fitted on training data only, then applied to test data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_miss = df_missing.drop(columns=["MedHouseVal"])
y_cls = (df["MedHouseVal"] > df["MedHouseVal"].median()).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X_miss, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

pipes = {
    "Median": Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    ),
    "KNN": Pipeline(
        [
            ("imp", KNNImputer(n_neighbors=5)),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    ),
    "MICE": Pipeline(
        [
            ("imp", IterativeImputer(random_state=42, max_iter=10)),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    ),
}

for name, pipe in pipes.items():
    scores = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring="accuracy")
    print(f"{name:>8s} Imputer -> CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 1.7 Handling Categorical Missing Values

In [ ]:
from sklearn.compose import ColumnTransformer

cat_data = pd.DataFrame(
    {
        "Color": np.random.choice(["Red", "Blue", "Green", None], size=200, p=[0.3, 0.3, 0.3, 0.1]),
        "Size": np.random.choice(["S", "M", "L", None], size=200, p=[0.25, 0.35, 0.25, 0.15]),
        "Weight": rng.normal(50, 10, 200),
    }
)
cat_data.loc[rng.random(200) < 0.08, "Weight"] = np.nan
print(cat_data.isna().sum())

In [ ]:
# Strategy 1: most_frequent for categorical, median for numeric
preprocessor = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median"), ["Weight"]),
        ("cat", SimpleImputer(strategy="most_frequent"), ["Color", "Size"]),
    ]
)
result = preprocessor.fit_transform(cat_data)
print("ColumnTransformer imputation (median + most_frequent):")
print(pd.DataFrame(result, columns=["Weight", "Color", "Size"]).head(10))

# Strategy 2: new "missing" category
cat_filled = cat_data.copy()
for col in ["Color", "Size"]:
    cat_filled[col] = cat_filled[col].fillna("missing")
print("\nValue counts after adding 'missing' category:")
print(cat_filled["Color"].value_counts())

## 1.8 Time Series Imputation

In [ ]:
dates = pd.date_range("2024-01-01", periods=50, freq="D")
values = np.sin(np.arange(50) * 0.3) * 10 + rng.normal(0, 1, 50) + 20
ts = pd.Series(values, index=dates)
ts.iloc[5:8] = np.nan
ts.iloc[20:24] = np.nan
ts.iloc[40] = np.nan

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
methods = {
    "Forward Fill (ffill)": ts.ffill(),
    "Backward Fill (bfill)": ts.bfill(),
    "Linear Interpolation": ts.interpolate(method="linear"),
    "Polynomial (order=3)": ts.interpolate(
        method="polynomial", order=3
    ),  # Fits a single polynomial of degree 3 (cubic) through the known points surrounding the gap, then reads off values from that curve.
    "Spline (order=3)": ts.interpolate(
        method="spline", order=3
    ),  # Fits a piecewise cubic spline (a series of connected cubic curves) through the known points, allowing for more flexibility in capturing local patterns around each gap.
    "Time-based": ts.interpolate(
        method="time"
    ),  # Uses the actual timestamps to perform interpolation, which can be more accurate for time series data with irregular intervals or non-linear trends.
}

for ax, (name, filled) in zip(axes.flat, methods.items()):
    ax.plot(ts.index, ts.values, "o-", label="Original", alpha=0.5, color="black")
    ax.plot(filled.index, filled.values, "s--", label=name, alpha=0.8, markersize=4)
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Time Series Imputation Methods", y=1.01)
plt.tight_layout()
plt.show()

# 2. Feature Engineering

Feature engineering is a **key stage of modeling** — the quality of the model depends on correct preprocessing. It happens in two moments:

1. **During ETL**: analytical engineering with domain knowledge (aggregations, statistics)
2. **After ETL**: creating additional variables or transforming existing ones to improve predictive power

**Important**: each transformation must be **fitted on training data only**, then applied to test data.

## 2.1 Numeric Variable Transformations

In [ ]:
from sklearn.model_selection import train_test_split

X = X_full.copy()
y = y_full.copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
feature_names = X.columns
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

### Min-Max Scaler
$$z = \frac{x - \min(x)}{\max(x) - \min(x)}$$
**Recommendation**: when the feature is more-or-less uniformly distributed across a fixed range.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
X_train_mm = minmax.fit_transform(X_train)
X_test_mm = minmax.transform(X_test)

# Custom range
minmax_custom = MinMaxScaler(feature_range=(-1, 1))
X_train_mm_c = minmax_custom.fit_transform(X_train)

print("Min-Max (0,1) — Train min:", X_train_mm.min(axis=0).round(3))
print("Min-Max (0,1) — Train max:", X_train_mm.max(axis=0).round(3))
print("Min-Max (0,1) — Test max: ", X_test_mm.max(axis=0).round(3), "(can exceed 1 on test!)")

### Standard Scaler (Z-Score)
$$z = \frac{x - \mu}{\sigma}$$
**Recommendation**: when the feature distribution does not contain extreme outliers.

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

print("Train mean:", X_train_std.mean(axis=0).round(6))
print("Train std: ", X_train_std.std(axis=0).round(6))
print("Learned means:", std_scaler.mean_.round(4))
print("Learned stds: ", std_scaler.scale_.round(4))

### Robust Scaler
Uses median and IQR — robust to outliers.
$$z = \frac{x - \text{median}(x)}{\text{IQR}(x)}$$
**Recommendation**: when the feature distribution contains extreme outliers.

In [ ]:
from sklearn.preprocessing import RobustScaler

robust = RobustScaler()
X_train_rob = robust.fit_transform(X_train)
print("Centers (medians):", robust.center_.round(4))
print("Scales (IQR):     ", robust.scale_.round(4))

### Clipping / Winsorization
If $x > \text{max}$, then $z = \text{max}$; if $x < \text{min}$, then $z = \text{min}$.

**Recommendation**: when the feature contains extreme outliers.

In [ ]:
feature = X_train["Population"].values
q01, q99 = np.percentile(feature, [1, 99])
clipped = np.clip(feature, q01, q99)

print(f"Original range: [{feature.min():.0f}, {feature.max():.0f}]")
print(f"Clipped [1%, 99%]: [{q01:.0f}, {q99:.0f}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(feature, bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Original Population")
axes[1].hist(clipped, bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_title("Clipped Population (1st-99th percentile)")
plt.tight_layout()
plt.show()

### Log Scaling
$$z = \log(x)$$ or $$z = \log(x + 1)$$ to handle zeros.
**Recommendation**: when the feature conforms to the power law (right-skewed).

In [ ]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log1p, inverse_func=np.expm1, validate=False)
X_train_log = log_transformer.fit_transform(X_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(X_train.iloc[:, 0], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title(f"Original {X.columns[0]}")
X_train_log_col = X_train_log.iloc[:, 0] if hasattr(X_train_log, "iloc") else X_train_log[:, 0]
axes[1].hist(X_train_log_col, bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_title(f"Log-transformed {X.columns[0]}")
plt.tight_layout()
plt.show()

### Quantile Transformer
Maps data to a uniform distribution (0-1) or normal distribution.

Use QuantileTransformer when your data has heavy outliers or severe skew, as it maps values by rank rather than magnitude — neutralizing extreme values and optionally reshaping the distribution to uniform or normal; avoid it when interpretability is important or your dataset is small.

In [ ]:
from sklearn.preprocessing import QuantileTransformer

qt_uniform = QuantileTransformer(output_distribution="uniform", n_quantiles=1000, random_state=42)
qt_normal = QuantileTransformer(output_distribution="normal", n_quantiles=1000, random_state=42)

X_train_qt_u = qt_uniform.fit_transform(X_train)
X_train_qt_n = qt_normal.fit_transform(X_train)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
idx = 4  # AveOccup (very skewed)
axes[0].hist(X_train.iloc[:, idx], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title(f"Original {X.columns[idx]}")
axes[1].hist(X_train_qt_u[:, idx], bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_title("Quantile (uniform)")
axes[2].hist(X_train_qt_n[:, idx], bins=50, edgecolor="black", alpha=0.7, color="teal")
axes[2].set_title("Quantile (normal)")
plt.tight_layout()
plt.show()

### Power Transformer (Yeo-Johnson, Box-Cox)
Maps data to as close to Gaussian as possible.
- **Box-Cox**: requires strictly positive data
- **Yeo-Johnson**: works with any data

In [ ]:
from sklearn.preprocessing import PowerTransformer

# Yeo-Johnson works with any data
pt_yj = PowerTransformer(method="yeo-johnson", standardize=True)
X_train_yj = pt_yj.fit_transform(X_train)
print("Yeo-Johnson lambdas:", pt_yj.lambdas_.round(4))

# Box-Cox requires strictly positive and non-constant data
# Select only columns that are strictly positive and have variance
X_train_pos = X_train.copy()
valid_cols = (X_train_pos.min() > 0) & (X_train_pos.nunique() > 1)
X_bc_input = X_train_pos.loc[:, valid_cols]
print(f"\nBox-Cox: {valid_cols.sum()} of {len(valid_cols)} features are strictly positive and non-constant")

pt_bc = PowerTransformer(method="box-cox", standardize=True)
X_train_bc = pt_bc.fit_transform(X_bc_input)
print("Box-Cox lambdas:", pt_bc.lambdas_.round(4))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
# Use first feature for comparison
axes[0].hist(X_train.iloc[:, 0], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title(f"Original {X.columns[0]}")
axes[1].hist(X_train_yj[:, 0], bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_title("Yeo-Johnson")
axes[2].hist(X_train_bc[:, 0], bins=50, edgecolor="black", alpha=0.7, color="teal")
axes[2].set_title("Box-Cox")
plt.tight_layout()
plt.show()

### Bucketing / Discretization
Dividing continuous variables into discrete bins. Strategies: uniform, quantile, kmeans.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

feature = X_train[["MedInc"]]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, strategy in zip(axes, ["uniform", "quantile", "kmeans"]):
    kbd = KBinsDiscretizer(n_bins=10, encode="ordinal", strategy=strategy)
    binned = kbd.fit_transform(feature)
    ax.hist(binned, bins=10, edgecolor="black", alpha=0.7)
    ax.set_title(f"KBinsDiscretizer ({strategy})")
    ax.set_xlabel("Bin")

plt.suptitle("Discretization of MedInc")
plt.tight_layout()
plt.show()

### Polynomial and Spline Transformations

#### Spline Transformer

Spline Transformer converts a single numerical feature into **multiple spline basis functions**, allowing linear models to capture non-linear relationships.

##### How it works

1. The feature range is divided into intervals using **knots** (boundary points)
2. Each interval gets its own smooth polynomial basis function
3. The original feature is replaced by **multiple columns** — one per basis function
4. A linear model trained on these columns can now fit curves, not just straight lines

##### Example

A single feature `age` with 5 knots becomes 6+ new columns. A linear regression on these columns can now model a curved relationship between age and the target.

##### When to use it

- You suspect a **non-linear relationship** between a feature and target
- You want to keep using a fast linear model (LogisticRegression, Ridge) without switching to a tree-based mode

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer

poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X_train[["MedInc", "AveRooms"]])
print(f"Polynomial (degree=2) from 2 features -> {X_poly.shape[1]} features")
print(f"Names: {poly.get_feature_names_out()}")

poly_inter = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_inter = poly_inter.fit_transform(X_train[["MedInc", "AveRooms"]])
print(f"\nInteraction-only from 2 features -> {X_inter.shape[1]} features")
print(f"Names: {poly_inter.get_feature_names_out()}")

spline = SplineTransformer(n_knots=5, degree=3, include_bias=False)
X_spline = spline.fit_transform(X_train[["MedInc"]])
print(f"\nSpline from 1 feature -> {X_spline.shape[1]} features")

### Visual Comparison of All Scalers

In [ ]:
from sklearn.preprocessing import MaxAbsScaler

original = X_train.iloc[:, 0].values.reshape(-1, 1)
transformers = {
    "Original": None,
    "MinMaxScaler": MinMaxScaler(),
    "StandardScaler": StandardScaler(),
    "RobustScaler": RobustScaler(),
    "MaxAbsScaler": MaxAbsScaler(),
    "Quantile\n(uniform)": QuantileTransformer(output_distribution="uniform", random_state=42),
    "Quantile\n(normal)": QuantileTransformer(output_distribution="normal", random_state=42),
    "PowerTransformer\n(Yeo-Johnson)": PowerTransformer(method="yeo-johnson"),
    "PowerTransformer\n(Box-Cox)": PowerTransformer(method="box-cox"),
}

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
for ax, (name, t) in zip(axes.flat, transformers.items()):
    data = original if t is None else t.fit_transform(original)
    ax.hist(data, bins=50, edgecolor="black", alpha=0.7)
    ax.set_title(name, fontsize=11)
plt.suptitle("All Numeric Transformations on MedInc", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2.2 Categorical Variable Transformations

### One-Hot Encoding
Transforms each categorical feature with $n$ categories into $n$ binary features.
- Drop one category to avoid perfect collinearity (for OLS without regularization)
- Can group infrequent categories

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_data = pd.DataFrame(
    {
        "Color": ["Red", "Blue", "Green", "Red", "Blue", "Yellow", "Red", "Green", "Red", "Red"],
        "Size": ["S", "M", "L", "M", "XL", "S", "M", "L", "M", "S"],
    }
)

# All categories
ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(cat_data)
print("All categories:")
print(pd.DataFrame(encoded, columns=ohe.get_feature_names_out()))

# Drop first (avoid collinearity)
ohe_drop = OneHotEncoder(sparse_output=False, drop="first")
encoded_drop = ohe_drop.fit_transform(cat_data)
print("\nDrop first:")
print(pd.DataFrame(encoded_drop, columns=ohe_drop.get_feature_names_out()))

# Group infrequent
# This handles rare categories in one-hot encoding by grouping them into a single "infrequent" bucket instead of giving each its own column.
# min_frequency=2 — any category that appears fewer than 2 times in training data gets treated as infrequent and merged into one _infrequent column.
# handle_unknown="infrequent_if_exist" — if a category appears at test time that was never seen in training, put it into the _infrequent column rather than raising an error.
ohe_infreq = OneHotEncoder(sparse_output=False, min_frequency=2, handle_unknown="infrequent_if_exist")
encoded_infreq = ohe_infreq.fit_transform(cat_data)
print("\nInfrequent grouped (min_frequency=2):")
print(pd.DataFrame(encoded_infreq, columns=ohe_infreq.get_feature_names_out()))
print(f"Infrequent: {ohe_infreq.infrequent_categories_}")

### Ordinal Encoder
Transforms categories to integers. **Caution**: imposes an order. Use only when categories have a natural order (e.g. low/medium/high).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# With explicit ordering
oe = OrdinalEncoder(categories=[["S", "M", "L", "XL"]])
encoded = oe.fit_transform(cat_data[["Size"]])
print("Ordinal (S=0 < M=1 < L=2 < XL=3):")
print(pd.DataFrame(encoded, columns=["Size_ordinal"]))

# Auto ordering
oe_auto = OrdinalEncoder()
encoded_auto = oe_auto.fit_transform(cat_data)
print("\nAuto ordering:")
print(pd.DataFrame(encoded_auto, columns=["Color_ord", "Size_ord"]))
print(f"Categories: {oe_auto.categories_}")

### Target Encoder
Encodes each category as the mean of the target for that category.
Uses internal cross-fitting to prevent target leakage.

In [ ]:
from sklearn.preprocessing import TargetEncoder

cat_train = pd.DataFrame(
    {
        "City": ["NYC", "LA", "NYC", "Chicago", "LA", "NYC", "Chicago", "LA", "NYC", "Chicago"] * 10,
        "Type": ["A", "B", "A", "B", "A", "B", "A", "B", "A", "A"] * 10,
    }
)
y_cat = np.array([100, 50, 110, 30, 55, 105, 35, 45, 95, 40] * 10) + np.random.normal(0, 5, 100)

te = TargetEncoder(smooth="auto", target_type="continuous")
encoded = te.fit_transform(cat_train, y_cat)
result = pd.DataFrame(encoded, columns=["City_te", "Type_te"])
result["City"] = cat_train["City"].values
result["y"] = y_cat
print("Target Encoded means:")
print(result.groupby("City")[["City_te", "y"]].mean().round(2))

### Label Encoder (for target variable only)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels = ["cat", "dog", "bird", "cat", "bird", "dog", "cat"]
encoded = le.fit_transform(labels)
print(f"Original: {labels}")
print(f"Encoded:  {encoded}")
print(f"Classes:  {le.classes_}")
print(f"Inverse:  {list(le.inverse_transform(encoded))}")

### ColumnTransformer for Mixed Data Types

Use ColumnTransformer to apply different transformations to numeric and categorical features in a single pipeline. This ensures proper handling of each feature type while maintaining a clean workflow.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

df_mixed = pd.DataFrame(
    {
        "Income": np.random.exponential(50000, 500),
        "Age": np.random.normal(40, 12, 500).clip(18, 80),
        "Education": np.random.choice(["HS", "BS", "MS", "PhD"], 500, p=[0.3, 0.4, 0.2, 0.1]),
        "City": np.random.choice(["NYC", "LA", "Chicago", "Houston"], 500),
    }
)
y_mixed = df_mixed["Income"] * 0.5 + df_mixed["Age"] * 100 + np.random.normal(0, 5000, 500)

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), ["Income", "Age"]),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), ["Education", "City"]),
    ]
)

pipe = Pipeline([("preprocess", preprocessor), ("model", LinearRegression())])
scores = cross_val_score(pipe, df_mixed, y_mixed, cv=5, scoring="r2")
print(f"Pipeline CV R2: {scores.mean():.4f} (+/- {scores.std():.4f})")
print(f"Feature names: {preprocessor.fit(df_mixed).get_feature_names_out()}")

## 2.3 Feature Interactions

We can look for interactions between variables (numeric & numeric, categorical & categorical, numeric & categorical) via multiplication, division, subtraction, etc.

In [ ]:
poly_interact = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_interact = poly_interact.fit_transform(X_train[["MedInc", "HouseAge", "AveRooms"]])
print(f"3 features -> {X_interact.shape[1]} interaction features")
print(f"Names: {poly_interact.get_feature_names_out()}")

# Manual feature engineering
df_eng = X_train.copy()
df_eng["Rooms_per_Person"] = df_eng["AveRooms"] / df_eng["AveOccup"]
df_eng["Income_x_Rooms"] = df_eng["MedInc"] * df_eng["AveRooms"]
df_eng["Age_squared"] = df_eng["HouseAge"] ** 2
df_eng["Log_Population"] = np.log1p(df_eng["Population"])
print(f"\nOriginal: {X_train.shape[1]} features -> Engineered: {df_eng.shape[1]} features")
print(f"New columns: {[c for c in df_eng.columns if c not in X_train.columns]}")

I recommend the following extra library for advanced categorical encoding methods: [category_encoders](https://contrib.scikit-learn.org/category_encoders/).

# 3. Regularization

Regularization modifies the loss function to incorporate information about parameter values — a tool to fight **overfitting** (excessive model variance):

$$\min_f \sum_{i=1}^{n} V(f(x_i), y_i) + \lambda R(f)$$

| Type | Name | Properties |
|------|------|-----------|
| **L1** | Lasso | Shrinks coefficients to exactly 0 — good for **feature selection** |
| **L2** | Ridge | Makes coefficients smaller — good for **multicollinearity** |
| **L1+L2** | Elastic Net | Combination — usually the most reasonable approach |

**For plain OLS**, scaling is *not* mathematically required — the solution is invariant to linear rescaling of individual features.

**For Ridge / Lasso / Elastic Net** (both linear and logistic), scaling is **essential**.

## 3.1 Comparing Regularization Strategies

In [ ]:
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    "OLS (no regularization)": LinearRegression(),
    "Ridge (L2, alpha=1.0)": Ridge(alpha=1.0),
    "Ridge (L2, alpha=10.0)": Ridge(alpha=10.0),
    "Lasso (L1, alpha=0.01)": Lasso(alpha=0.01),
    "Lasso (L1, alpha=0.1)": Lasso(alpha=0.1),
    "ElasticNet (a=0.01, l1=0.5)": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "ElasticNet (a=0.1, l1=0.7)": ElasticNet(alpha=0.1, l1_ratio=0.7),
}

print(f"{'Model':<35} {'CV R2':>8} {'Std':>8}")
print("-" * 55)
for name, model in models.items():
    scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring="r2")
    print(f"{name:<35} {scores.mean():>8.4f} {scores.std():>8.4f}")

## 3.2 Lasso Feature Selection Effect

In [ ]:
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]

fig, ax = plt.subplots(figsize=(14, 6))
coef_matrix = []
for alpha in alphas:
    lasso = Lasso(alpha=alpha).fit(X_train_s, y_train)
    coef_matrix.append(lasso.coef_)
    n_zero = (lasso.coef_ == 0).sum()
    print(f"alpha={alpha:<6} -> {n_zero} zero coefficients: {lasso.coef_.round(4)}")

coef_matrix = np.array(coef_matrix)
for i in range(coef_matrix.shape[1]):
    ax.plot(alphas, coef_matrix[:, i], "o-", label=feature_names[i])

ax.set_xscale("log")
ax.set_xlabel("Alpha")
ax.set_ylabel("Coefficient")
ax.set_title("Lasso: Coefficients vs Alpha")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.axhline(y=0, color="black", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

## 3.3 Regularization Paths: Lasso vs Ridge vs Elastic Net

In [ ]:
from sklearn.linear_model import enet_path, lasso_path

alphas_grid = np.logspace(-4, 1, 50)

alphas_lasso, coefs_lasso, _ = lasso_path(X_train_s, y_train, alphas=alphas_grid)
alphas_enet, coefs_enet, _ = enet_path(X_train_s, y_train, l1_ratio=0.5, alphas=alphas_grid)

ridge_coefs = []
ridge_alphas = np.logspace(-4, 4, 50)
for a in ridge_alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_train_s, y_train).coef_)
ridge_coefs = np.array(ridge_coefs)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for i in range(coefs_lasso.shape[0]):
    axes[0].plot(alphas_lasso, coefs_lasso[i], label=feature_names[i])
axes[0].set_xscale("log")
axes[0].set_title("Lasso (L1)")
axes[0].set_xlabel("Alpha")

for i in range(ridge_coefs.shape[1]):
    axes[1].plot(ridge_alphas, ridge_coefs[:, i], label=feature_names[i])
axes[1].set_xscale("log")
axes[1].set_title("Ridge (L2)")
axes[1].set_xlabel("Alpha")

for i in range(coefs_enet.shape[0]):
    axes[2].plot(alphas_enet, coefs_enet[i], label=feature_names[i])
axes[2].set_xscale("log")
axes[2].set_title("Elastic Net")
axes[2].set_xlabel("Alpha")

for ax in axes:
    ax.axhline(y=0, color="black", linestyle="--", alpha=0.3)
    ax.legend(fontsize=7, bbox_to_anchor=(1.0, 1))
plt.suptitle("Regularization Paths")
plt.tight_layout()
plt.show()

## 3.4 Automatic Alpha Selection with CV

In [ ]:
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV

ridge_cv = RidgeCV(alphas=np.logspace(-4, 4, 50), cv=5, scoring="r2")
ridge_cv.fit(X_train_s, y_train)
print(f"RidgeCV     -> best alpha: {ridge_cv.alpha_:.6f}, test R2: {ridge_cv.score(X_test_s, y_test):.4f}")

lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, random_state=42)
lasso_cv.fit(X_train_s, y_train)
print(f"LassoCV     -> best alpha: {lasso_cv.alpha_:.6f}, test R2: {lasso_cv.score(X_test_s, y_test):.4f}")

enet_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], alphas=np.logspace(-4, 1, 50), cv=5, random_state=42)
enet_cv.fit(X_train_s, y_train)
r2 = enet_cv.score(X_test_s, y_test)
print(f"ElasticNetCV -> best alpha: {enet_cv.alpha_:.6f}, l1_ratio: {enet_cv.l1_ratio_:.2f}, test R2: {r2:.4f}")

## 3.5 Regularization for Classification

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target
)
sc = StandardScaler()
Xc_tr_s = sc.fit_transform(Xc_tr)
Xc_te_s = sc.transform(Xc_te)

classifiers = {
    "LogReg (no penalty)": LogisticRegression(penalty=None, max_iter=5000),
    "LogReg (L2, C=1)": LogisticRegression(penalty="l2", C=1.0, max_iter=5000),
    "LogReg (L2, C=0.01)": LogisticRegression(penalty="l2", C=0.01, max_iter=5000),
    "LogReg (L1, C=1)": LogisticRegression(penalty="l1", C=1.0, solver="saga", max_iter=5000),
    "LogReg (ElasticNet)": LogisticRegression(penalty="elasticnet", C=1.0, l1_ratio=0.5, solver="saga", max_iter=5000),
    "LogReg (L1, C=100)": LogisticRegression(penalty="l1", C=100.0, solver="saga", max_iter=5000),
    "LogReg (L2, C=100)": LogisticRegression(penalty="l2", C=100.0, max_iter=5000),
    "LogReg (ElasticNet, C=100)": LogisticRegression(
        penalty="elasticnet", C=100.0, l1_ratio=0.5, solver="saga", max_iter=5000
    ),
}

print(f"{'Model':<30} {'Accuracy':>10} {'Non-zero':>10}")
print("-" * 53)
for name, clf in classifiers.items():
    clf.fit(Xc_tr_s, yc_tr)
    print(f"{name:<30} {clf.score(Xc_te_s, yc_te):>10.4f} {np.sum(clf.coef_ != 0):>10}")

# 4. Feature Selection

After feature engineering, we should have many variables. Not all models can select variables themselves (e.g. OLS, KNN, SVM). Recommended **two-step approach**:

1. **Univariate/bivariate**: Variance Threshold, Mutual Information, F-test, Chi2, Correlation
2. **Multivariate**: Embedded (Lasso, Boruta, SHAP), Wrapper (Backward, Forward, RFE)

## 4.1 Univariate Feature Selection

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import (
    SelectKBest,
    SelectPercentile,
    VarianceThreshold,
    chi2,
    f_classif,
    mutual_info_classif,
)

cancer = load_breast_cancer()
X_sel = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_sel = cancer.target

print(f"Original features: {X_sel.shape[1]}")

### Variance Threshold
Removes features with variance below a threshold.

In [ ]:
vt = VarianceThreshold(threshold=0.5)
X_vt = vt.fit_transform(X_sel)
removed = X_sel.columns[~vt.get_support()]
print(f"Variance Threshold (>0.5): {X_sel.shape[1]} -> {X_vt.shape[1]} features")
print(f"Removed: {list(removed)}")

# Show variances of all features
variances = pd.Series(X_sel.var(), name="Variance").sort_values()
print("\nFeatures with lowest variance:")
print(variances.head(10).round(4))

### F-test (ANOVA) and Chi-squared Test

Both methods rank features by their statistical relationship to the target **without training a model**, then select the top k.

- **F-test (ANOVA)** measures whether a feature's mean differs significantly between classes.
A high F-score means the feature separates classes well; a low p-value confirms the difference is not due to chance.
Works with any continuous features.

 - **Chi-squared** measures whether a feature and the target are statistically independent.
A high score means the feature and class label co-occur more than expected by chance.
Requires non-negative values — scale with MinMaxScaler first if needed.

| x | F-test | Chi-squared |
|---|---|---|
| Measures | Difference in means between classes | Statistical dependence |
| Input | Continuous features | Non-negative values (Originally designed for count/frequency data (e.g. word counts), but works on scaled continuous features too) |

**Limitation:** both methods score each feature in isolation — they cannot detect interactions between features.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# F-test for classification
# Asks: "Does the mean of this feature differ significantly between classes?"
selector_f = SelectKBest(score_func=f_classif, k=10)
X_f = selector_f.fit_transform(X_sel, y_sel)
f_scores = pd.DataFrame(
    {
        "Feature": cancer.feature_names,
        "F-score": selector_f.scores_,
        "p-value": selector_f.pvalues_,
        "Selected": selector_f.get_support(),
    }
).sort_values("F-score", ascending=False)
print("Top 10 by F-test:")
print(f_scores.head(10).to_string(index=False))

# Chi2 (requires non-negative features)
# Asks: "Is this feature statistically independent of the target?"
X_pos = MinMaxScaler().fit_transform(
    X_sel
)  # Requires non-negative values — that's why MinMaxScaler is applied first (scales to [0,1])
selector_chi2 = SelectKBest(score_func=chi2, k=10)
selector_chi2.fit_transform(X_pos, y_sel)
chi2_scores = pd.DataFrame(
    {"Feature": cancer.feature_names, "Chi2-score": selector_chi2.scores_, "Selected": selector_chi2.get_support()}
).sort_values("Chi2-score", ascending=False)
print("\nTop 10 by Chi2:")
print(chi2_scores.head(10).to_string(index=False))

### Mutual Information
Measures mutual dependence between variables (related to entropy).
Formula: $$I(X; Y) = H(X) - H(X|Y)$$
- $H(X)$: entropy of feature X (uncertainty)
- $H(X|Y)$: conditional entropy of X given Y (uncertainty remaining about X after knowing Y)

A high mutual information score means knowing the feature reduces uncertainty about the target, indicating a strong relationship.

In [ ]:
selector_mi = SelectKBest(score_func=mutual_info_classif, k=10)
selector_mi.fit(X_sel, y_sel)
mi_scores = pd.DataFrame(
    {"Feature": cancer.feature_names, "MI-score": selector_mi.scores_, "Selected": selector_mi.get_support()}
).sort_values("MI-score", ascending=False)
print("Top 10 by Mutual Information:")
print(mi_scores.head(10).to_string(index=False))

### SelectPercentile
Select features based on percentile of the highest scores.

In [ ]:
selector_pct = SelectPercentile(score_func=f_classif, percentile=30)
X_pct = selector_pct.fit_transform(X_sel, y_sel)
print(f"SelectPercentile (top 30%): {X_sel.shape[1]} -> {X_pct.shape[1]} features")
print(f"Selected: {list(X_sel.columns[selector_pct.get_support()])}")

### Correlation Analysis

Three correlation methods measure the relationship between a feature and the target,
each suited to different data characteristics.

- **Pearson** measures linear relationship between two continuous variables.
Assumes both variables are normally distributed and the relationship is a straight line.
Use when: data is continuous, roughly normal, and you expect a linear relationship.

- **Spearman** measures monotonic relationship using ranks instead of raw values.
Does not assume normality — works on the ranked order of values, not their magnitude.
Use when: data is skewed, has outliers, or the relationship is monotonic but not strictly linear.

- **Kendall** also measures monotonic relationship via ranks, but counts concordant vs discordant pairs.
More robust than Spearman on small samples and less sensitive to tied values.
Use when: dataset is small or contains many tied values.

| Criteria | Pearson | Spearman | Kendall |
|---|---|---|---|
| Relationship type | Linear | Monotonic | Monotonic |
| Assumes normality | Yes | No | No |
| Uses ranks | No | Yes | Yes |
| Best for | Large, normal, continuous data | Skewed or outlier-prone data | Small samples or tied values |

All three return values between -1 and 1: 1 = perfect positive, -1 = perfect negative, 0 = no relationship.

In [ ]:
# Pearson, Spearman, Kendall correlations with target
corr_methods = {}
for method in ["pearson", "spearman", "kendall"]:
    corr = X_sel.apply(lambda col: col.corr(pd.Series(y_sel), method=method))
    corr_methods[method] = corr.abs()

corr_df = pd.DataFrame(corr_methods).sort_values("spearman", ascending=False)
print("Top 10 features by absolute correlation with target:")
print(corr_df.head(10).round(4))

# Feature-feature correlation heatmap (to detect multicollinearity)
fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = X_sel.corr(method="spearman")
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0,
    ax=ax,
    fmt=".1f",
    annot=True,
    annot_kws={"size": 6},
    linewidths=0.5,
)
ax.set_title("Feature-Feature Spearman Correlation")
plt.tight_layout()
plt.show()

## 4.2 Multivariate Feature Selection

### Recursive Feature Elimination (RFE)

RFE selects features by repeatedly training a model and removing the least important feature each round.

**How it works:**
1. Train the model on all features
2. Rank features by importance (coefficients or feature importances)
3. Remove the least important feature
4. Repeat until the desired number of features remains

**Key advantage over statistical filters (F-test, MI):**
RFE uses an actual model to judge importance, so it captures feature interactions —
a feature that is weak alone may still be selected if it complements other features.

**Trade-off:** significantly slower than filter methods — it trains the model
`n_features - n_features_to_select` times.

> Works with any model that exposes `coef_` (e.g. LogisticRegression, Ridge, Lasso, linear SVM).


In [ ]:
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression

# RFE with Logistic Regression
lr = LogisticRegression(max_iter=5000, C=1.0, random_state=42)
scaler_sel = StandardScaler()
X_sel_s = scaler_sel.fit_transform(X_sel)

rfe = RFE(estimator=lr, n_features_to_select=10, step=1)
rfe.fit(X_sel_s, y_sel)
print("RFE selected features (top 10):")
selected = pd.DataFrame(
    {"Feature": cancer.feature_names, "Selected": rfe.support_, "Ranking": rfe.ranking_}
).sort_values("Ranking")
print(selected.head(15).to_string(index=False))

### RFECV (RFE with Cross-Validation)

RFECV extends RFE by using cross-validation to automatically find the **optimal number of features**
— you no longer need to specify `n_features_to_select` manually.

**How it works:**
1. Run RFE for every possible number of features (from 1 to all)
2. At each step, evaluate model performance using k-fold cross-validation
3. Select the feature count that produced the best CV score

**Key advantage over RFE:**
RFE requires you to guess how many features to keep. RFECV finds that number for you
by measuring actual model performance at each step.

**Trade-off:** even slower than RFE — it trains the model
`n_features × n_folds` times.

> Use RFECV when you don't know how many features to keep.
> Use RFE when you already have a target feature count and need faster results.


In [ ]:
rfecv = RFECV(
    estimator=lr, step=1, cv=5, scoring="accuracy", min_features_to_select=2
)  # step=1 — remove 1 feature per round; higher values (e.g. step=5) are faster but less precise
rfecv.fit(X_sel_s, y_sel)
print(f"Optimal number of features: {rfecv.n_features_}")
print(f"Selected features: {list(cancer.feature_names[rfecv.support_])}")

fig, ax = plt.subplots(figsize=(10, 5))
scores_mean = rfecv.cv_results_["mean_test_score"]
scores_std = rfecv.cv_results_["std_test_score"]
n_features_range = range(2, len(scores_mean) + 2)
ax.plot(n_features_range, scores_mean, "o-")
ax.fill_between(n_features_range, scores_mean - scores_std, scores_mean + scores_std, alpha=0.2)
ax.set_xlabel("Number of features")
ax.set_ylabel("CV Accuracy")
ax.set_title("RFECV: Accuracy vs Number of Features")
ax.axvline(x=rfecv.n_features_, color="red", linestyle="--", label=f"Optimal: {rfecv.n_features_}")
ax.legend()
plt.tight_layout()
plt.show()

### Sequential Feature Selection (Forward & Backward)

SFS builds the feature set one step at a time by greedily adding or removing features
based on cross-validated model performance.

**Forward selection** starts with zero features and adds the best one each round:
1. Try each remaining feature individually
2. Add whichever improves CV score the most
3. Repeat until the desired number of features is reached

**Backward selection** starts with all features and removes the worst one each round:
1. Try removing each feature one at a time
2. Drop whichever causes the least drop in CV score
3. Repeat until the desired number of features is reached

**Key difference from RFE:**
RFE ranks and removes by model coefficients/importances.
SFS tries every candidate at each step and picks by actual CV score — slower but more direct.

| x | Forward | Backward |
|---|---|---|
| Starts with | No features | All features |
| Risk | May miss interactions only visible with more features | Slow on very high-dimensional data |
| Prefer when | Many irrelevant features | Most features are useful |


In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

# Forward selection
sfs_forward = SequentialFeatureSelector(lr, n_features_to_select=10, direction="forward", cv=5, scoring="accuracy")
sfs_forward.fit(X_sel_s, y_sel)
print(f"Forward selection (10 features): {list(cancer.feature_names[sfs_forward.get_support()])}")

# Backward selection
sfs_backward = SequentialFeatureSelector(lr, n_features_to_select=10, direction="backward", cv=5, scoring="accuracy")
sfs_backward.fit(X_sel_s, y_sel)
print(f"Backward selection (10 features): {list(cancer.feature_names[sfs_backward.get_support()])}")

### Embedded: SelectFromModel (Lasso-based)

In [ ]:
from sklearn.feature_selection import SelectFromModel

lasso = Lasso(alpha=0.01).fit(X_sel_s, y_sel)
sfm = SelectFromModel(lasso, prefit=True)
X_sfm = sfm.transform(X_sel_s)
print(f"SelectFromModel (Lasso, alpha=0.01): {X_sel.shape[1]} -> {X_sfm.shape[1]} features")
print(f"Selected: {list(cancer.feature_names[sfm.get_support()])}")

I recommend the following additional library for advanced feature selection methods that complement scikit-learn: [feature-engine](https://feature-engine.trainindata.com/en/latest/).

# 5. Classes Rebalancing

Imbalanced classes negatively impact the cost function (the algorithm focuses on the majority class) and mislead evaluation metrics like Accuracy.

Techniques:
1. **Modification of the cost function** (class weights)
2. **Undersampling** (reduce majority class)
3. **Oversampling** (increase minority class)
4. **Combination** (over + under)

**Warning**: rebalancing must be done **separately for each fold** in cross-validation to avoid leakage!

## 5.1 Creating an Imbalanced Dataset

In [ ]:
from sklearn.datasets import make_classification

X_imb, y_imb = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    weights=[0.95, 0.05],
    random_state=42,
    flip_y=0.01,
)

print(f"Class distribution: {dict(zip(*np.unique(y_imb, return_counts=True)))}")
print(f"Imbalance ratio: {np.sum(y_imb == 0) / np.sum(y_imb == 1):.1f}:1")

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb)

## 5.2 Cost Function Modification (Class Weights)

$$L(X; \theta) = \sum_i W_{y_i} L(x_i; \theta) \quad \text{where} \quad W_c = \frac{N}{\text{number of samples of class } c}$$

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Without class weights
lr_no_w = LogisticRegression(max_iter=1000, random_state=42)
lr_no_w.fit(Xi_tr, yi_tr)
print("=== Without class weights ===")
print(classification_report(yi_te, lr_no_w.predict(Xi_te)))

# With class weights
lr_w = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_w.fit(Xi_tr, yi_tr)
print("=== With class_weight='balanced' ===")
print(classification_report(yi_te, lr_w.predict(Xi_te)))

## 5.3 Undersampling

Reducing the number of observations from the majority class.

### Prototype generation: Cluster Centroids
Uses K-means to synthesize new majority class samples at cluster centers.

### Prototype selection: Random Undersampler, Tomek Links, Edited Nearest Neighbours

**RandomUnderSampler** — Randomly removes observations from the majority class to balance class distribution, by default to a 1:1 ratio. It is the simplest and fastest method, but carries the risk of losing important information since samples are discarded without considering their significance.

**ClusterCentroids** — Replaces majority class observations with centroids computed by the K-Means algorithm, creating synthetic points representing clusters instead of randomly discarding samples. The method preserves the underlying data structure but is computationally expensive and sensitive to feature scaling.

**TomekLinks** — Removes so-called Tomek links, i.e. pairs of nearest neighbors belonging to different classes, by eliminating the majority-class observation from each pair. Rather than balancing the classes, this method cleans the decision boundary of noisy and borderline observations, improving class separability.

**EditedNearestNeighbours (ENN)** — Removes majority class observations whose label disagrees with the majority of their k nearest neighbors (k=3 by default). Like TomekLinks, its goal is data cleaning — removing difficult and potentially mislabeled samples — rather than aggressive class balancing.

**RepeatedEditedNearestNeighbours (RENN)** — Repeatedly applies the ENN procedure until no more observations are removed or a specified number of iterations is reached. The iterative approach results in more rigorous cleaning and a smoother decision boundary, at the cost of greater data loss.

**NearMiss (version=1)** — Selects majority class observations that have the smallest average distance to their k nearest minority class observations. Unlike cleaning-based methods, NearMiss retains samples near the class boundary, which can help the classifier but also increases sensitivity to noise around the decision boundary.

In [ ]:
from imblearn.under_sampling import (
    ClusterCentroids,
    EditedNearestNeighbours,
    NearMiss,
    RandomUnderSampler,
    RepeatedEditedNearestNeighbours,
    TomekLinks,
)

undersamplers = {
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
    "RandomUnderSampler (0.5)": RandomUnderSampler(sampling_strategy=0.5, random_state=42),
    "ClusterCentroids": ClusterCentroids(random_state=42),
    "TomekLinks": TomekLinks(),
    "EditedNearestNeighbours": EditedNearestNeighbours(),
    "RepeatedEditedNN": RepeatedEditedNearestNeighbours(),
    "NearMiss (v1)": NearMiss(version=1),
}

print(f"{'Method':<30} {'Majority':>10} {'Minority':>10} {'Ratio':>8}")
print("-" * 62)
print(
    f"{'Original':<30} {np.sum(yi_tr == 0):>10} {np.sum(yi_tr == 1):>10} {np.sum(yi_tr == 0) / np.sum(yi_tr == 1):>8.1f}"
)
for name, sampler in undersamplers.items():
    X_res, y_res = sampler.fit_resample(Xi_tr, yi_tr)
    print(
        f"{name:<30} {np.sum(y_res == 0):>10} {np.sum(y_res == 1):>10} {np.sum(y_res == 0) / max(np.sum(y_res == 1), 1):>8.1f}"
    )

## 5.4 Oversampling

Increasing the number of observations from the minority class.

### Random Oversampler, SMOTE, Borderline SMOTE, SVM SMOTE, KMeans SMOTE, ADASYN

**RandomOverSampler** — Randomly duplicates existing observations from the minority class until the class distribution is balanced. It is simple and fast, but because it creates exact copies rather than new samples, it can easily lead to overfitting on the duplicated points.

**SMOTE (Synthetic Minority Over-sampling Technique)** — Generates synthetic minority class samples by interpolating between a minority observation and one of its k nearest minority neighbors. Unlike random oversampling, it creates genuinely new points and reduces overfitting, but it can blur class boundaries by generating samples in overlapping regions.

**BorderlineSMOTE** — A variant of SMOTE that generates synthetic samples only near the decision boundary, i.e. from minority observations whose neighbors are mostly from the majority class. This focuses the oversampling effort on the "hard" examples that the classifier struggles with most, rather than wasting it on already well-classified regions.

**SVMSMOTE** — Uses a Support Vector Machine to identify support vectors in the minority class, and generates synthetic samples along the decision boundary defined by those vectors. This approach produces samples in regions most relevant for classification, but is more computationally expensive and sensitive to SVM hyperparameters.

**KMeansSMOTE** — First applies K-Means clustering, then performs SMOTE within clusters that have a high proportion of minority samples (controlled by `cluster_balance_threshold`). This avoids generating noisy samples in majority-dominated regions and helps address within-class imbalance, i.e. rare sub-concepts inside the minority class.

**ADASYN (Adaptive Synthetic Sampling)** — Similar to SMOTE, but generates more synthetic samples for minority observations that are harder to learn — those surrounded by many majority class neighbors. This adaptively shifts the classifier's focus toward difficult regions, though it can also amplify noise if outliers are mistaken for hard examples.

In [ ]:
from imblearn.over_sampling import ADASYN, SMOTE, SVMSMOTE, BorderlineSMOTE, KMeansSMOTE, RandomOverSampler

oversamplers = {
    "RandomOverSampler": RandomOverSampler(random_state=42),
    "SMOTE": SMOTE(random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=42),
    "SVMSMOTE": SVMSMOTE(random_state=42),
    "KMeansSMOTE": KMeansSMOTE(random_state=42, cluster_balance_threshold=0.01),
    "ADASYN": ADASYN(random_state=42),
}

print(f"{'Method':<25} {'Majority':>10} {'Minority':>10} {'Ratio':>8}")
print("-" * 57)
print(
    f"{'Original':<25} {np.sum(yi_tr == 0):>10} {np.sum(yi_tr == 1):>10} {np.sum(yi_tr == 0) / np.sum(yi_tr == 1):>8.1f}"
)
for name, sampler in oversamplers.items():
    X_res, y_res = sampler.fit_resample(Xi_tr, yi_tr)
    print(
        f"{name:<25} {np.sum(y_res == 0):>10} {np.sum(y_res == 1):>10} {np.sum(y_res == 0) / max(np.sum(y_res == 1), 1):>8.1f}"
    )

## 5.5 Combination: SMOTE + Cleaning

SMOTE can generate noisy samples. Cleaning with Tomek Links or ENN removes overlap.

In [ ]:
from imblearn.combine import SMOTEENN, SMOTETomek

combiners = {
    "SMOTETomek": SMOTETomek(random_state=42),
    "SMOTEENN": SMOTEENN(random_state=42),
}

print(f"{'Method':<20} {'Majority':>10} {'Minority':>10}")
print("-" * 44)
print(f"{'Original':<20} {np.sum(yi_tr == 0):>10} {np.sum(yi_tr == 1):>10}")
for name, sampler in combiners.items():
    X_res, y_res = sampler.fit_resample(Xi_tr, yi_tr)
    print(f"{name:<20} {np.sum(y_res == 0):>10} {np.sum(y_res == 1):>10}")

## 5.6 Visualizing Resampling Effects (2D)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(Xi_tr)

methods_to_plot = {
    "Original": (X_2d, yi_tr),
    "Random Under": RandomUnderSampler(random_state=42).fit_resample(Xi_tr, yi_tr),
    "SMOTE": SMOTE(random_state=42).fit_resample(Xi_tr, yi_tr),
    "ADASYN": ADASYN(random_state=42).fit_resample(Xi_tr, yi_tr),
    "SMOTETomek": SMOTETomek(random_state=42).fit_resample(Xi_tr, yi_tr),
    "SMOTEENN": SMOTEENN(random_state=42).fit_resample(Xi_tr, yi_tr),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (name, data) in zip(axes.flat, methods_to_plot.items()):
    if name == "Original":
        X_plot, y_plot = data
    else:
        X_plot = pca.transform(data[0])
        y_plot = data[1]
    ax.scatter(X_plot[y_plot == 0, 0], X_plot[y_plot == 0, 1], alpha=0.3, s=10, label="Majority")
    ax.scatter(X_plot[y_plot == 1, 0], X_plot[y_plot == 1, 1], alpha=0.6, s=20, label="Minority")
    ax.set_title(f"{name} (maj={np.sum(y_plot == 0)}, min={np.sum(y_plot == 1)})")
    ax.legend(fontsize=8)

plt.suptitle("Effect of Resampling Techniques (PCA 2D)")
plt.tight_layout()
plt.show()

## 5.7 Proper Pipeline with Resampling (imblearn Pipeline)

Use `imblearn.pipeline.Pipeline` instead of `sklearn.pipeline.Pipeline` for resampling steps.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipelines = {
    "No resampling": ImbPipeline(
        [
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "Class weights": ImbPipeline(
        [
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ]
    ),
    "Random Under": ImbPipeline(
        [
            ("smp", RandomUnderSampler(random_state=42)),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "SMOTE": ImbPipeline(
        [
            ("smp", SMOTE(random_state=42)),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "SMOTETomek": ImbPipeline(
        [
            ("smp", SMOTETomek(random_state=42)),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "SMOTEENN": ImbPipeline(
        [
            ("smp", SMOTEENN(random_state=42)),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
}

print(f"{'Method':<20} {'CV F1 (macro)':>15} {'Std':>8}")
print("-" * 47)
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, Xi_tr, yi_tr, cv=cv, scoring="f1_macro")
    print(f"{name:<20} {scores.mean():>15.4f} {scores.std():>8.4f}")

# 6. Ensemble Methods

Ensemble methods combine multiple weak models to improve performance:
1. **Bagging**: Bootstrap aggregating — trains multiple base models on bootstrap samples
2. **Stacking**: A meta-learner combines outputs of base models
3. **Voting**: Majority vote or averaging from multiple models


## 6.1 Bagging

Bagging (Bootstrap Aggregating) reduces variance by training multiple models on different
random subsets of the training data, then combining their predictions.

**How it works:**
1. Draw k random samples **with replacement** (bootstrap) from the training set
2. Train a separate model on each sample
3. Aggregate predictions — majority vote for classification, average for regression

Because each model sees a slightly different dataset, they make different errors.
Averaging over many models cancels out individual mistakes, producing a more stable prediction.

**Key property:** works best with **high-variance models** (e.g. KNN, deep decision trees)
that are sensitive to small changes in training data. Bagging stabilizes them without
increasing bias.

> Bagging does NOT help with high-bias models — if the base model systematically
> underfits, averaging many underfitting models still underfits.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

cancer = load_breast_cancer()
Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target
)
sc_e = StandardScaler()
Xe_tr_s = sc_e.fit_transform(Xe_tr)
Xe_te_s = sc_e.transform(Xe_te)

bagging_models = {
    "BaggingClassifier (10 trees)": BaggingClassifier(
        estimator=KNeighborsClassifier(n_neighbors=5), n_estimators=10, random_state=42
    ),
    "BaggingClassifier (50 trees)": BaggingClassifier(
        estimator=KNeighborsClassifier(n_neighbors=5), n_estimators=50, random_state=42
    ),
}

print(f"{'Model':<35} {'CV Accuracy':>12} {'Std':>8}")
print("-" * 58)
for name, model in bagging_models.items():
    scores = cross_val_score(model, Xe_tr_s, ye_tr, cv=5, scoring="accuracy")
    print(f"{name:<35} {scores.mean():>12.4f} {scores.std():>8.4f}")

## 6.2 Stacking

A meta-learner is trained to combine predictions of base models.

**How it works:**
1. Train several diverse base models (e.g. KNN, LogisticRegression, SVM) on the training data
2. Use their **out-of-fold predictions** as input features for a second-level model (the meta-learner)
3. The meta-learner learns how to best weight and combine the base model outputs

Out-of-fold predictions are used to prevent the meta-learner from overfitting —
each base model predicts on data it was not trained on.

**Key advantage over Bagging and Voting:**
Instead of treating all models equally, the meta-learner **learns** which models to trust
and in which situations — capturing complementary strengths of different algorithms.

**Typical choice for meta-learner:** a simple model like LogisticRegression or Ridge,
to avoid overfitting on top of the base predictions.

> Stacking tends to give the best accuracy of all ensemble methods,
> but is the most complex to implement and slowest to train.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Define base estimators
base_estimators = [
    ("lr", LogisticRegression(max_iter=5000, random_state=42)),
    ("svc", SVC(probability=True, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
]

# Stacking with Logistic Regression as meta-learner
stacking = StackingClassifier(
    estimators=base_estimators, final_estimator=LogisticRegression(max_iter=5000, random_state=42), cv=5
)

scores = cross_val_score(stacking, Xe_tr_s, ye_tr, cv=5, scoring="accuracy")
print(f"Stacking (3 base + LR meta): CV Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

# Compare with individual models
for name, model in base_estimators:
    scores_i = cross_val_score(model, Xe_tr_s, ye_tr, cv=5, scoring="accuracy")
    print(f"  {name}: {scores_i.mean():.4f} (+/- {scores_i.std():.4f})")

## 6.3 Voting

Combine predictions from multiple models via hard or soft voting.

**Hard voting** — each model casts a class vote, the majority wins.
Works with any classifier regardless of whether it outputs probabilities.

**Soft voting** — average the predicted probabilities across models, pick the class with the highest mean probability.
Generally more accurate than hard voting as it accounts for prediction confidence,
but requires all models to support `predict_proba`.

**Example:**

| Model | P(class=1) | Hard vote |
|---|---|---|
| LogisticRegression | 0.70 | 1 |
| KNN | 0.60 | 1 |
| SVM | 0.35 | 0 |

- Hard voting → class 1 (2 vs 1)
- Soft voting → mean = 0.55 → class 1 (but with less confidence than if all agreed)

**When to use voting over stacking:**
simpler to implement, no risk of meta-learner overfitting, and often competitive
when base models are already diverse and well-tuned.


In [ ]:
from sklearn.ensemble import VotingClassifier

# Hard voting
voting_hard = VotingClassifier(estimators=base_estimators, voting="hard")

# Soft voting (requires probability support)
voting_soft = VotingClassifier(estimators=base_estimators, voting="soft")

# Weighted soft voting
voting_weighted = VotingClassifier(
    estimators=base_estimators,
    voting="soft",
    weights=[2, 1, 1],  # give more weight to LogisticRegression
)

for name, voter in [("Hard voting", voting_hard), ("Soft voting", voting_soft), ("Weighted soft", voting_weighted)]:
    scores = cross_val_score(voter, Xe_tr_s, ye_tr, cv=5, scoring="accuracy")
    print(f"{name:<20}: CV Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

## 6.4 Ensemble Comparison Summary

In [ ]:
from sklearn.ensemble import (
    BaggingClassifier,
    StackingClassifier,
    VotingClassifier,
)

all_models = {
    "Bagging": BaggingClassifier(estimator=KNeighborsClassifier(n_neighbors=5), n_estimators=50, random_state=42),
    "Voting (soft)": voting_soft,
    "Stacking": stacking,
}

results = {}
for name, model in all_models.items():
    scores = cross_val_score(model, Xe_tr_s, ye_tr, cv=5, scoring="accuracy")
    results[name] = scores

fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(results.values(), labels=results.keys())
ax.set_ylabel("CV Accuracy")
ax.set_title("Ensemble Methods Comparison")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 7. Probability Calibration

Probability calibration allows to:
1. Better calibrate probabilities of a classification model
2. Add probability support for models that don't return it natively (e.g. SVM)

**Well calibrated classifiers**: the output probability can be directly interpreted as a confidence level.

Useful in credit risk, insurance, and any domain where correctly estimated probabilities matter.

## 7.1 Calibration Curves (Reliability Diagrams)

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

X_cal, y_cal = make_classification(n_samples=5000, n_features=20, n_informative=10, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_cal, y_cal, test_size=0.3, random_state=42, stratify=y_cal)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (Platt)": SVC(probability=True, random_state=42),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, model in models.items():
    model.fit(Xc_tr, yc_tr)
    CalibrationDisplay.from_estimator(model, Xc_te, yc_te, n_bins=10, name=name, ax=axes[0])

axes[0].set_title("Calibration Curves (before calibration)")

# Show distribution of predicted probabilities
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(Xc_te)[:, 1]
    else:
        probs = model.decision_function(Xc_te)
    axes[1].hist(probs, bins=30, alpha=0.4, label=name)
axes[1].set_title("Distribution of Predicted Probabilities")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7.2 Calibrating Models

Calibration fits a regressor (calibrator) that maps model output to calibrated probabilities in [0, 1].

**Important**: must be done via cross-validation to avoid bias.

Two calibration methods:
- **Sigmoid** (Platt scaling): fits $p = 1/(1 + \exp(Af + B))$ — best when model is under-confident
- **Isotonic**: non-parametric monotone fit — more powerful but prone to overfitting (needs >1000 samples)

In [ ]:
# Calibrate each model with sigmoid and isotonic methods
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax_row, method in zip(axes, ["sigmoid", "isotonic"]):
    for ax, (name, model_cls) in zip(
        ax_row,
        [
            ("KNN (k=5)", KNeighborsClassifier(n_neighbors=5)),
            ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42)),
        ],
    ):
        # Uncalibrated
        model_cls.fit(Xc_tr, yc_tr)
        CalibrationDisplay.from_estimator(model_cls, Xc_te, yc_te, n_bins=10, name=f"{name} (uncalibrated)", ax=ax)

        # Calibrated
        calibrated = CalibratedClassifierCV(model_cls, method=method, cv=5)
        calibrated.fit(Xc_tr, yc_tr)
        CalibrationDisplay.from_estimator(calibrated, Xc_te, yc_te, n_bins=10, name=f"{name} ({method})", ax=ax)

        ax.set_title(f"{name} --- {method} calibration")

plt.suptitle("Effect of Probability Calibration")
plt.tight_layout()
plt.show()

## 7.3 Brier Score

The Brier score measures the quality of calibration (lower is better):
$$BS = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)^2$$

In [ ]:
from sklearn.metrics import brier_score_loss

print(f"{'Model':<35} {'Brier (uncal)':>14} {'Brier (sig)':>12} {'Brier (iso)':>12}")
print("-" * 78)

for name, model_cls in [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
    ("KNN (k=5)", KNeighborsClassifier(n_neighbors=5)),
    ("SVC", SVC(random_state=42)),
]:
    # Uncalibrated
    model_cls.fit(Xc_tr, yc_tr)
    if hasattr(model_cls, "predict_proba"):
        p_uncal = model_cls.predict_proba(Xc_te)[:, 1]
        bs_uncal = brier_score_loss(yc_te, p_uncal)
    else:
        bs_uncal = float("nan")

    # Sigmoid calibration
    cal_sig = CalibratedClassifierCV(model_cls, method="sigmoid", cv=5)
    cal_sig.fit(Xc_tr, yc_tr)
    p_sig = cal_sig.predict_proba(Xc_te)[:, 1]
    bs_sig = brier_score_loss(yc_te, p_sig)

    # Isotonic calibration
    cal_iso = CalibratedClassifierCV(model_cls, method="isotonic", cv=5)
    cal_iso.fit(Xc_tr, yc_tr)
    p_iso = cal_iso.predict_proba(Xc_te)[:, 1]
    bs_iso = brier_score_loss(yc_te, p_iso)

    print(f"{name:<35} {bs_uncal:>14.6f} {bs_sig:>12.6f} {bs_iso:>12.6f}")

## 7.4 Calibration in a Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

# SVM doesn't return probabilities by default - calibration adds this capability
pipe_svm_calibrated = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", CalibratedClassifierCV(SVC(random_state=42), method="sigmoid", cv=5)),
    ]
)

pipe_svm_calibrated.fit(Xc_tr, yc_tr)
probs = pipe_svm_calibrated.predict_proba(Xc_te)
print(f"SVM with calibration - predict_proba shape: {probs.shape}")
print(f"Sample predictions: {probs[:5].round(4)}")
print(f"Brier score: {brier_score_loss(yc_te, probs[:, 1]):.6f}")

# 8. Model Depreciation — Pattern Change in ML

Models degrade over time as data distributions and relationships change. Three types of drift:

| Drift Type | Description | Features-Target Dynamics |
|-----------|-------------|-------------------------|
| **Data Drift** | Change in distribution of input features | Remains the same |
| **Concept Drift** | Change in relationship between features and target | Changes over time |
| **Covariate Drift** | Change in distribution of *some* input features | Remains the same |

**Examples**:
- Data drift: shift in house size distribution, changes in transaction amounts
- Concept drift: spam characteristics evolve, new fraud patterns, new money laundering techniques
- Covariate drift: shift in customer age distribution in churn prediction

## 8.1 Simulating and Detecting Data Drift

In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

# Simulate data drift: training data vs shifted "production" data
np.random.seed(42)
X_reference = np.random.normal(0, 1, (1000, 5))
X_drifted = np.random.normal(0.5, 1.2, (1000, 5))  # shifted mean and variance
X_no_drift = np.random.normal(0.02, 1.01, (1000, 5))  # nearly identical

feature_names_drift = [f"Feature_{i}" for i in range(5)]

print("=== Drift Detection: KS Test ===")
print(f"{'Feature':<15} {'Drifted (stat)':>15} {'p-value':>10} {'No-drift (stat)':>16} {'p-value':>10}")
print("-" * 70)
for i, name in enumerate(feature_names_drift):
    stat_d, p_d = ks_2samp(X_reference[:, i], X_drifted[:, i])
    stat_n, p_n = ks_2samp(X_reference[:, i], X_no_drift[:, i])
    print(f"{name:<15} {stat_d:>15.4f} {p_d:>10.6f} {stat_n:>16.4f} {p_n:>10.6f}")

In [ ]:
# Visualize distribution drift
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, data) in zip(
    axes,
    [
        ("Reference", X_reference),
        ("Drifted (mean shift + var change)", X_drifted),
        ("No Drift", X_no_drift),
    ],
):
    for i in range(min(3, data.shape[1])):
        ax.hist(data[:, i], bins=30, alpha=0.4, label=feature_names_drift[i])
    ax.set_title(title)
    ax.legend()

plt.suptitle("Detecting Data Drift via Distribution Comparison")
plt.tight_layout()
plt.show()

## 8.2 Simulating Concept Drift

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Simulate concept drift: relationship between X and y changes over time
np.random.seed(42)

# Period 1: y = 2*x1 + 3*x2 + noise
X_p1 = np.random.normal(0, 1, (500, 2))
y_p1 = 2 * X_p1[:, 0] + 3 * X_p1[:, 1] + np.random.normal(0, 0.5, 500)

# Period 2: y = -1*x1 + 5*x2 + noise (relationship changed!)
X_p2 = np.random.normal(0, 1, (500, 2))
y_p2 = -1 * X_p2[:, 0] + 5 * X_p2[:, 1] + np.random.normal(0, 0.5, 500)

# Train on period 1, test on period 1 vs period 2
model_p1 = LinearRegression().fit(X_p1, y_p1)

pred_p1 = model_p1.predict(X_p1)
pred_p2 = model_p1.predict(X_p2)

print("Model trained on Period 1:")
print(f"  Coefficients: {model_p1.coef_.round(4)}")
print(f"  Period 1 R2: {r2_score(y_p1, pred_p1):.4f}, MSE: {mean_squared_error(y_p1, pred_p1):.4f}")
print(f"  Period 2 R2: {r2_score(y_p2, pred_p2):.4f}, MSE: {mean_squared_error(y_p2, pred_p2):.4f}")
print("  -> Concept drift causes severe performance degradation!")

# Retrained model on period 2
model_p2 = LinearRegression().fit(X_p2, y_p2)
print("\nRetrained on Period 2:")
print(f"  Coefficients: {model_p2.coef_.round(4)}")
print(f"  Period 2 R2: {r2_score(y_p2, model_p2.predict(X_p2)):.4f}")

## 8.3 Monitoring Model Performance Over Time

In [ ]:
# Simulate model performance degradation over time
np.random.seed(42)
n_periods = 12
train_data = np.random.normal(0, 1, (1000, 5))
train_target = 2 * train_data[:, 0] - train_data[:, 1] + np.random.normal(0, 0.5, 1000)

model_monitor = LinearRegression().fit(train_data, train_target)

performances = []
for period in range(n_periods):
    drift = period * 0.1  # gradually increasing drift
    test_data = np.random.normal(drift, 1 + drift * 0.1, (200, 5))
    # Also gradually shift the relationship
    test_target = (2 - drift * 0.15) * test_data[:, 0] - test_data[:, 1] + np.random.normal(0, 0.5, 200)
    r2 = r2_score(test_target, model_monitor.predict(test_data))
    mse = mean_squared_error(test_target, model_monitor.predict(test_data))
    performances.append({"Period": period + 1, "R2": r2, "MSE": mse})

perf_df = pd.DataFrame(performances)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(perf_df["Period"], perf_df["R2"], "o-", color="teal")
axes[0].axhline(y=0.8, color="red", linestyle="--", label="Acceptable threshold")
axes[0].set_xlabel("Time Period")
axes[0].set_ylabel("R2 Score")
axes[0].set_title("Model R2 Over Time (Drift)")
axes[0].legend()

axes[1].plot(perf_df["Period"], perf_df["MSE"], "o-", color="coral")
axes[1].set_xlabel("Time Period")
axes[1].set_ylabel("MSE")
axes[1].set_title("Model MSE Over Time (Drift)")

plt.suptitle("Model Depreciation Monitoring")
plt.tight_layout()
plt.show()

## 8.4 Wasserstein Distance for Drift Detection

In [ ]:
print("=== Wasserstein Distance ===")
print(f"{'Feature':<15} {'Drifted':>10} {'No-drift':>10}")
print("-" * 38)
for i, name in enumerate(feature_names_drift):
    w_d = wasserstein_distance(X_reference[:, i], X_drifted[:, i])
    w_n = wasserstein_distance(X_reference[:, i], X_no_drift[:, i])
    print(f"{name:<15} {w_d:>10.4f} {w_n:>10.4f}")

print("\nLarger Wasserstein distance = more drift")

**Recommendation NannyML**
(https://github.com/nannyml/nannyml) is a strong choice for monitoring machine learning models in production, especially when true target labels are delayed or unavailable.

# 9. Model Evaluation & Diagnostics

A good model is not just a high-accuracy model. You also need:

1. The **right metric** for your task (regression / classification / probability).
2. A read on the **bias–variance trade-off** — is the model under- or overfitting?
3. An honest picture of **generalization** — how does performance change with more data or different hyperparameters?

This section covers the full catalogue of metrics needed to evaluate regression and classification models, plus empirical learning curves that visualize under- and overfitting.

## 9.1 Regression Metrics Side-by-Side

Let $\hat{y}_i$ denote predictions and $y_i$ denote true values, with $\bar{y}$ the mean of $y$.

| Metric | Formula | Notes |
|---|---|---|
| **MSE** | $\tfrac{1}{n}\sum (y_i - \hat{y}_i)^2$ | Penalizes large errors heavily; unit = target² |
| **RMSE** | $\sqrt{\text{MSE}}$ | Same unit as target; still outlier-sensitive |
| **MAE** | $\tfrac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | Robust to outliers; everybody’s favorite baseline |
| **MedAE** | $\text{median}(\lvert y_i - \hat{y}_i \rvert)$ | Extremely robust — ignores tail |
| **MAPE** | $\tfrac{100\%}{n}\sum \tfrac{\lvert y_i - \hat{y}_i \rvert}{\lvert y_i \rvert}$ | Scale-free (%) but breaks for $y_i \approx 0$ |
| **sMAPE** | $\tfrac{100\%}{n}\sum \tfrac{2\lvert y_i - \hat{y}_i \rvert}{\lvert y_i \rvert + \lvert \hat{y}_i \rvert}$ | Symmetric MAPE, bounded in $[0, 200\%]$ |
| **MSLE** | $\tfrac{1}{n}\sum(\log(1+y_i) - \log(1+\hat{y}_i))^2$ | Penalizes **relative** (not absolute) error — good for skewed targets |
| **R²** | $1 - \tfrac{\sum (y_i-\hat y_i)^2}{\sum (y_i-\bar y)^2}$ | Fraction of variance explained; can be **negative** for bad models |
| **Adj. R²** | $1 - (1-R^2)\tfrac{n-1}{n-p-1}$ | Penalizes extra features |

**Which to use?**
* RMSE: default for “punish big mistakes more”.
* MAE: default for “all errors equal” / robust.
* MAPE / sMAPE: when you need a **percentage** report and targets are strictly positive.
* MSLE: when targets span orders of magnitude (prices, counts).
* R²: communicating variance-explained to non-technical stakeholders.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    mean_squared_log_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# Load regression data (subsample for speed)
housing = fetch_california_housing()
X_r, y_r = housing.data[:5000], housing.target[:5000]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.3, random_state=42)
scaler_r = StandardScaler()
X_train_rs = scaler_r.fit_transform(X_train_r)
X_test_rs = scaler_r.transform(X_test_r)


def smape(y_true, y_pred):
    """Symmetric MAPE in [0, 200] (%) — handles y=0 gracefully."""
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true)[mask] / denom[mask])


def adjusted_r2(y_true, y_pred, n_features):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)


models_r = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=1)": Ridge(alpha=1.0),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01),
    "KNN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "SVR (RBF)": SVR(kernel="rbf", C=1.0),
}

rows = []
for name, m in models_r.items():
    m.fit(X_train_rs, y_train_r)
    yp = m.predict(X_test_rs)
    rows.append(
        {
            "Model": name,
            "MSE": mean_squared_error(y_test_r, yp),
            "RMSE": np.sqrt(mean_squared_error(y_test_r, yp)),
            "MAE": mean_absolute_error(y_test_r, yp),
            "MAPE (%)": mean_absolute_percentage_error(y_test_r, yp) * 100,
            "sMAPE (%)": smape(y_test_r, yp),
            "MSLE": mean_squared_log_error(y_test_r, np.clip(yp, 0, None)),
            "MedAE": median_absolute_error(y_test_r, yp),
            "R^2": r2_score(y_test_r, yp),
            "Adj R^2": adjusted_r2(y_test_r, yp, X_test_rs.shape[1]),
        }
    )

metrics_reg_df = pd.DataFrame(rows).round(4)
print(metrics_reg_df.to_string(index=False))

# Note: the Dummy model should produce R^2 ~ 0 and the worst RMSE —
# if you see better, something is wrong with your evaluation.

In [ ]:
# Visual side-by-side of a few error-type metrics for each model
error_metrics = ["RMSE", "MAE", "MedAE"]

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.25
x = np.arange(len(metrics_reg_df))
for i, met in enumerate(error_metrics):
    ax.bar(x + i * width, metrics_reg_df[met].values, width, label=met, edgecolor="black")
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_reg_df["Model"], rotation=20)
ax.set_ylabel("Error (target units)")
ax.set_title("Regression error metrics across models", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Observation: RMSE > MAE > MedAE always (by Jensen-style inequalities).")
print("A big gap between RMSE and MAE hints at heavy-tailed residuals / outliers.")

## 9.2 Classification Metrics Beyond Accuracy

On **imbalanced data** accuracy is actively misleading — predicting "all negative" on a 99:1 dataset gives 99% accuracy.
The confusion matrix is the starting point:

| x          | Pred 0         | Pred 1         |
|------------|----------------|----------------|
| **True 0** | TN             | FP (Type I)    |
| **True 1** | FN (Type II)   | TP             |

From this we derive:

| Metric | Formula | Intuition |
|---|---|---|
| **Accuracy** | $(TP+TN)/N$ | Overall hit rate — bad on imbalanced data |
| **Precision (PPV)** | $TP/(TP+FP)$ | Of those I called positive, how many really are? |
| **Recall (sensitivity, TPR)** | $TP/(TP+FN)$ | Of all true positives, how many did I catch? |
| **Specificity (TNR)** | $TN/(TN+FP)$ | Of all true negatives, how many did I correctly reject? |
| **NPV** | $TN/(TN+FN)$ | Of those I called negative, how many really are? |
| **Fβ-score** | $(1+\beta^2)\cdot\tfrac{\text{prec}\cdot\text{rec}}{\beta^2\,\text{prec}+\text{rec}}$ | Harmonic-weighted mix — $\beta>1$ prioritizes recall, $\beta<1$ prioritizes precision |
| **MCC** | $\tfrac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$ | Correlation coefficient on the confusion matrix; $\in [-1, 1]$, robust to imbalance |

**When to prefer what:**
* Medical screening / fraud: you care about **recall** (don't miss positives) → F$_2$.
* Search / recommendation: you care about **precision** (top-k is good) → F$_{0.5}$.
* Balanced overall quality on imbalanced data: **MCC** is often the single best scalar.

In [ ]:
from collections import Counter

from imblearn.datasets import make_imbalance
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    fbeta_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

# Load + imbalance to showcase why accuracy is misleading
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_c, y_c = make_imbalance(X_c, y_c, sampling_strategy={0: 40, 1: 357}, random_state=42)
print("Class distribution:", Counter(y_c))

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.3, random_state=42, stratify=y_c)
scaler_c = StandardScaler()
X_train_cs = scaler_c.fit_transform(X_train_c)
X_test_cs = scaler_c.transform(X_test_c)

clf = LogisticRegression(max_iter=5000).fit(X_train_cs, y_train_c)
y_pred_c = clf.predict(X_test_cs)
y_prob_c = clf.predict_proba(X_test_cs)[:, 1]

# Confusion matrix and derived rates
tn, fp, fn, tp = confusion_matrix(y_test_c, y_pred_c).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)
npv = tn / (tn + fn)

metrics_clf = {
    "Accuracy": accuracy_score(y_test_c, y_pred_c),
    "Balanced Accuracy": balanced_accuracy_score(y_test_c, y_pred_c),
    "Precision (PPV)": ppv,
    "Recall / Sensitivity (TPR)": sensitivity,
    "Specificity (TNR)": specificity,
    "NPV": npv,
    "F_0.5 (precision-heavy)": fbeta_score(y_test_c, y_pred_c, beta=0.5),
    "F_1 (balanced)": fbeta_score(y_test_c, y_pred_c, beta=1.0),
    "F_2 (recall-heavy)": fbeta_score(y_test_c, y_pred_c, beta=2.0),
    "MCC": matthews_corrcoef(y_test_c, y_pred_c),
}
print("\nConfusion matrix:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")
print("\nMetric side-by-side:")
for k, v in metrics_clf.items():
    print(f"  {k:<26s}  {v:.4f}")

# Double-check that sklearn's recall_score / precision_score match our manual ones
assert np.isclose(recall_score(y_test_c, y_pred_c), sensitivity)
assert np.isclose(precision_score(y_test_c, y_pred_c), ppv)

In [ ]:
# Compare many classifiers with MCC + F-beta on the imbalanced dataset
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

clf_zoo = {
    "DummyClassifier (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Logistic Regression (balanced)": LogisticRegression(max_iter=5000, class_weight="balanced"),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)": SVC(kernel="rbf", C=1.0, probability=True, random_state=42),
}

rows_c = []
for name, m in clf_zoo.items():
    m.fit(X_train_cs, y_train_c)
    yp = m.predict(X_test_cs)
    rows_c.append(
        {
            "Model": name,
            "Acc": accuracy_score(y_test_c, yp),
            "Bal. Acc": balanced_accuracy_score(y_test_c, yp),
            "F_0.5": fbeta_score(y_test_c, yp, beta=0.5, zero_division=0),
            "F_1": fbeta_score(y_test_c, yp, beta=1.0, zero_division=0),
            "F_2": fbeta_score(y_test_c, yp, beta=2.0, zero_division=0),
            "MCC": matthews_corrcoef(y_test_c, yp),
        }
    )

metrics_clf_df = pd.DataFrame(rows_c).round(4)
print(metrics_clf_df.to_string(index=False))

print("\nNote: DummyClassifier can hit ~90% accuracy here by always predicting the")
print("majority class — but MCC is 0 (no information!) and recall on the minority is 0.")

## 9.3 Probability Metrics: ROC, PR, Log-loss, Gini, Lift

When your classifier outputs probabilities you get much richer evaluation tools. Instead of committing to a single 0.5 threshold, you can sweep the threshold and measure *how well the model ranks positives above negatives*.

### ROC and AUC-ROC
* **ROC curve**: TPR vs FPR as the decision threshold varies.
* **AUC-ROC**: area under the ROC curve. Probability that a random positive scores higher than a random negative.
* **Gini coefficient**: $2 \cdot \text{AUC-ROC} - 1$ — popular in credit scoring.

### Precision–Recall and AUC-PR
* **PR curve**: Precision vs Recall. Much more informative than ROC on **imbalanced** data.
* **AUC-PR / Average Precision**: area under the PR curve. Baseline = prevalence of the positive class.

### Log-loss (Binary Cross-Entropy)
* $\text{log-loss} = -\tfrac{1}{n}\sum_i \bigl[y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\bigr]$
* Rewards **well-calibrated** probabilities. A model that is 90% confident and wrong is punished severely.

### Lift curve
Popular in marketing / churn:
* Sort records by predicted probability, descending.
* Cumulative lift at top $q\%$ = (positives in top $q\%$) / (positives in a random $q\%$).
* Lift of 3 at the top decile means “calling the top 10% gets us 3× more positives than calling randomly”.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# Compare a good model, a mediocre model, and a random one
clf_good = SVC(kernel="rbf", C=1.0, probability=True, random_state=42).fit(X_train_cs, y_train_c)
clf_mid = LogisticRegression(max_iter=5000).fit(X_train_cs, y_train_c)

probs = {
    "SVM (RBF)": clf_good.predict_proba(X_test_cs)[:, 1],
    "Logistic Regression": clf_mid.predict_proba(X_test_cs)[:, 1],
    "Random": np.random.RandomState(0).rand(len(y_test_c)),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
baseline_pr = y_test_c.mean()

for name, p in probs.items():
    # ROC
    fpr, tpr, _ = roc_curve(y_test_c, p)
    auc_roc = roc_auc_score(y_test_c, p)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_roc:.3f})")

    # PR
    prec, rec, _ = precision_recall_curve(y_test_c, p)
    ap = average_precision_score(y_test_c, p)
    axes[1].plot(rec, prec, lw=2, label=f"{name} (AP = {ap:.3f})")

    # Lift
    order = np.argsort(-p)
    y_sorted = y_test_c[order]
    cum_pos = np.cumsum(y_sorted)
    ks = np.arange(1, len(y_sorted) + 1)
    lift = (cum_pos / ks) / baseline_pr
    axes[2].plot(ks / len(y_sorted) * 100, lift, lw=2, label=name)

axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curves", fontweight="bold")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)

axes[1].axhline(baseline_pr, ls="--", color="gray", label=f"baseline = {baseline_pr:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curves", fontweight="bold")
axes[1].legend(loc="lower left")
axes[1].grid(True, alpha=0.3)

axes[2].axhline(1.0, ls="--", color="gray", label="random = 1×")
axes[2].set_xlabel("Top x% of predictions (by score)")
axes[2].set_ylabel("Cumulative lift")
axes[2].set_title("Lift curves", fontweight="bold")
axes[2].legend(loc="upper right")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Numeric summary
print(f"{'Model':<25} {'AUC-ROC':>10} {'Gini':>10} {'AUC-PR':>10} {'Log-loss':>12}")
print("-" * 70)
for name, p in probs.items():
    auc_roc = roc_auc_score(y_test_c, p)
    ap = average_precision_score(y_test_c, p)
    ll = log_loss(y_test_c, np.clip(p, 1e-9, 1 - 1e-9))
    print(f"{name:<25} {auc_roc:>10.4f} {2 * auc_roc - 1:>10.4f} {ap:>10.4f} {ll:>12.4f}")

## 9.4 Bias–Variance Trade-off via Learning Curves

Every model’s error can be decomposed as

$$\mathbb{E}\bigl[(y - \hat{f}(x))^2\bigr] \;=\; \underbrace{\bigl(\mathbb{E}[\hat{f}(x)] - f(x)\bigr)^2}_{\text{bias}^2} \;+\; \underbrace{\mathrm{Var}\bigl[\hat{f}(x)\bigr]}_{\text{variance}} \;+\; \underbrace{\sigma^2}_{\text{irreducible noise}}$$

* **High bias** (underfitting): model is too simple — training AND validation error are both high and close together.
* **High variance** (overfitting): model memorizes training data — training error is low, validation error is much higher.
* **Sweet spot**: the gap closes and both errors plateau at a low level.

Two empirical tools diagnose this:

| Tool | What varies on x-axis | Diagnoses |
|---|---|---|
| **Learning curve** | Training set **size** | Would more data help? Is the model under/overfitting? |
| **Validation curve** | A single **hyperparameter** | Optimal hyperparameter and bias–variance regime |

In [ ]:
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.neighbors import KNeighborsRegressor

# ----- Learning curves for models of very different capacity -----
models_lc = {
    "Ridge (simple)": Ridge(alpha=10.0),
    "KNN k=1 (flexible)": KNeighborsRegressor(n_neighbors=1),
}

train_sizes = np.linspace(0.1, 1.0, 8)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (name, est) in zip(axes, models_lc.items()):
    sizes, train_sc, val_sc = learning_curve(
        est,
        X_train_rs,
        y_train_r,
        train_sizes=train_sizes,
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        random_state=42,
    )
    # Convert to positive MSE
    train_mse = -train_sc
    val_mse = -val_sc
    ax.plot(sizes, train_mse.mean(axis=1), "o-", label="Training MSE")
    ax.fill_between(
        sizes,
        train_mse.mean(axis=1) - train_mse.std(axis=1),
        train_mse.mean(axis=1) + train_mse.std(axis=1),
        alpha=0.15,
    )
    ax.plot(sizes, val_mse.mean(axis=1), "s-", label="Validation MSE")
    ax.fill_between(
        sizes, val_mse.mean(axis=1) - val_mse.std(axis=1), val_mse.mean(axis=1) + val_mse.std(axis=1), alpha=0.15
    )
    ax.set_xlabel("Training samples")
    ax.set_ylabel("MSE")
    ax.set_title(f"Learning curve: {name}", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Read the shape:")
print("• Curves close together & plateaued high -> HIGH BIAS. Try a more flexible model.")
print("• Big persistent gap -> HIGH VARIANCE. Regularize, more data, simpler model.")
print("• Curves converging low -> you are near the sweet spot.")

In [ ]:
# ----- Validation curve: MSE vs a single hyperparameter -----
alpha_range = np.logspace(-3, 4, 15)

train_sc_vc, val_sc_vc = validation_curve(
    Ridge(),
    X_train_rs,
    y_train_r,
    param_name="alpha",
    param_range=alpha_range,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
train_mse = -train_sc_vc
val_mse = -val_sc_vc

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(alpha_range, train_mse.mean(axis=1), "o-", label="Training MSE")
ax.fill_between(
    alpha_range,
    train_mse.mean(axis=1) - train_mse.std(axis=1),
    train_mse.mean(axis=1) + train_mse.std(axis=1),
    alpha=0.15,
)
ax.semilogx(alpha_range, val_mse.mean(axis=1), "s-", label="Validation MSE")
ax.fill_between(
    alpha_range, val_mse.mean(axis=1) - val_mse.std(axis=1), val_mse.mean(axis=1) + val_mse.std(axis=1), alpha=0.15
)
ax.set_xlabel("alpha (log scale)")
ax.set_ylabel("MSE")
ax.set_title("Validation curve: Ridge alpha sweep", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_alpha = alpha_range[np.argmin(val_mse.mean(axis=1))]
print(f"Best alpha by validation MSE: {best_alpha:.4f}")
print("Left side (small alpha): training << validation -> overfitting (high variance).")
print("Right side (large alpha): both high and close -> underfitting (high bias).")

# 10. Cross-Validation Variants

5-fold / 10-fold K-Fold is the default, but it is far from the only option.
Sklearn ships with a zoo of cross-validation splitters, each appropriate for different data shapes / constraints.

| Splitter | Use when… |
|---|---|
| `KFold`, `StratifiedKFold` | Standard i.i.d. data |
| `LeaveOneOut` (LOO) | Very small datasets and low-variance estimates needed |
| `LeavePOut` | Small datasets, combinatorial study |
| `RepeatedKFold`, `RepeatedStratifiedKFold` | Small/medium data — want tighter variance on the CV score |
| `GroupKFold` / `LeaveOneGroupOut` | Observations are grouped (e.g. multiple rows per patient) |
| `TimeSeriesSplit` | Temporal data — you must never train on the future |
| **Nested CV** (any two splitters) | Hyperparameter tuning + unbiased generalization estimate |

## 10.1 `LeaveOneOut`

LOO leaves a **single sample** out in each fold. With $n$ samples you do $n$ folds.

* Pro: the lowest-bias CV estimate you can get.
* Con: high variance; extremely slow for anything with $n > \text{a few hundred}$.
* Practical use: tiny datasets, some theoretical analyses.

In [ ]:
from sklearn.model_selection import LeaveOneOut, LeavePOut, cross_val_score

# Use a small subset (LOO is O(n) model fits)
X_small = X_train_rs[:100]
y_small = y_train_r[:100]

loo = LeaveOneOut()
print(f"LOO folds on n={len(X_small)}: {loo.get_n_splits(X_small)}")

scores = cross_val_score(
    Ridge(alpha=1.0),
    X_small,
    y_small,
    cv=loo,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
print(f"LOO mean MSE: {-scores.mean():.4f}   (std across folds is {scores.std():.4f})")
print("Note: LOO fold scores are 0/1-like for classification (one sample), so averaging smooths things out.")

## 10.2 `LeavePOut`

LeavePOut leaves out all possible subsets of $p$ samples — so the number of folds is $\binom{n}{p}$.
Useful for tiny datasets and for combinatorial cross-validation studies. Exploding cost past $n \sim 30$.

In [ ]:
from math import comb

# Drop to 25 samples — otherwise the # of folds is enormous
X_tiny = X_train_rs[:25]
y_tiny = y_train_r[:25]

lpo = LeavePOut(p=2)
print(
    f"LeavePOut(p=2) on n={len(X_tiny)}: {lpo.get_n_splits(X_tiny)} folds "
    f"(= C({len(X_tiny)}, 2) = {comb(len(X_tiny), 2)})"
)

scores = cross_val_score(
    Ridge(alpha=1.0),
    X_tiny,
    y_tiny,
    cv=lpo,
    scoring="r2",
    n_jobs=-1,
)
print(f"LeavePOut mean R^2: {scores.mean():.4f}   std: {scores.std():.4f}")
print("\nRule of thumb: if LeavePOut(p=5) would give > 1000 folds, pick RepeatedKFold instead.")

## 10.3 `RepeatedKFold` and `RepeatedStratifiedKFold`

Repeats K-Fold CV $R$ times with different random seeds and concatenates the results.
You get $R\cdot K$ fold scores instead of $K$ — **tighter variance on the CV estimate** at the cost of more compute.
For classification with class imbalance use `RepeatedStratifiedKFold`.

In [ ]:
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold

# Regression (RepeatedKFold)
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
scores_rkf = cross_val_score(Ridge(alpha=1.0), X_train_rs, y_train_r, cv=rkf, scoring="r2", n_jobs=-1)
print(f"RepeatedKFold(5 x 10) -> {scores_rkf.size} fold scores")
print(
    f"  Mean R^2: {scores_rkf.mean():.4f}   Std: {scores_rkf.std():.4f}   "
    f"95% CI (empirical): [{np.percentile(scores_rkf, 2.5):.4f}, {np.percentile(scores_rkf, 97.5):.4f}]"
)

# Classification (RepeatedStratifiedKFold)
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scores_rskf = cross_val_score(
    LogisticRegression(max_iter=5000), X_train_cs, y_train_c, cv=rskf, scoring="roc_auc", n_jobs=-1
)
print(f"\nRepeatedStratifiedKFold(5 x 10) -> {scores_rskf.size} fold scores")
print(f"  Mean AUC: {scores_rskf.mean():.4f}   Std: {scores_rskf.std():.4f}")

# Visualize score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(scores_rkf, bins=20, edgecolor="black", alpha=0.7, color="tab:blue")
axes[0].axvline(scores_rkf.mean(), color="red", ls="--", label=f"mean = {scores_rkf.mean():.4f}")
axes[0].set_title("RepeatedKFold R^2 distribution (regression)")
axes[0].legend()
axes[1].hist(scores_rskf, bins=20, edgecolor="black", alpha=0.7, color="tab:orange")
axes[1].axvline(scores_rskf.mean(), color="red", ls="--", label=f"mean = {scores_rskf.mean():.4f}")
axes[1].set_title("RepeatedStratifiedKFold AUC distribution (classification)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 10.4 Nested Cross-Validation

If you tune hyperparameters on the same CV you use to report performance, your reported score is **optimistically biased** — you essentially “overfit” to the validation folds. **Nested CV** separates these two concerns:

* **Outer loop**: estimates the expected generalization performance of the *whole pipeline* (including tuning).
* **Inner loop**: searches for the best hyperparameters on each outer training fold.

```
for outer_train, outer_test in outer_cv.split(X):
    inner_search = GridSearchCV(model, param_grid, cv=inner_cv).fit(X[outer_train], y[outer_train])
    score_on_this_outer_fold = inner_search.score(X[outer_test], y[outer_test])
```

Result: an unbiased estimate, but **very** expensive (e.g. a 5-outer × 3-inner grid of 10 configurations runs the model 150 times).

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

inner_cv = KFold(n_splits=3, shuffle=True, random_state=0)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {"alpha": np.logspace(-3, 3, 7)}
inner_search = GridSearchCV(Ridge(), param_grid, cv=inner_cv, scoring="r2", n_jobs=-1)

# Nested CV: one cross_val_score call does it all
nested_scores = cross_val_score(
    inner_search,
    X_train_rs,
    y_train_r,
    cv=outer_cv,
    scoring="r2",
    n_jobs=-1,
)

# Non-nested CV (the biased comparison)
inner_search.fit(X_train_rs, y_train_r)
non_nested_score = inner_search.best_score_

print(f"Non-nested  (biased) CV R^2: {non_nested_score:.4f}")
print(f"Nested CV R^2:               {nested_scores.mean():.4f} ± {nested_scores.std():.4f}")
print("\nTypically: non-nested > nested (the non-nested estimate is optimistically biased).")

## 10.5 `TimeSeriesSplit`

For temporal data, a random K-Fold split is **cheating** — you would train on the future and test on the past. `TimeSeriesSplit` makes each fold’s training set a **prefix** of the series and its test set the **next chunk**. Past never leaks into past.

Two variants:
* **Expanding window** (default): training set keeps growing.
* **Rolling window** (`max_train_size`): training set stays the same size.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

n_samples = 100
tscv_expanding = TimeSeriesSplit(n_splits=5)
tscv_rolling = TimeSeriesSplit(n_splits=5, max_train_size=20)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, tscv, title in [
    (axes[0], tscv_expanding, "TimeSeriesSplit — EXPANDING window"),
    (axes[1], tscv_rolling, "TimeSeriesSplit — ROLLING window (max_train_size=20)"),
]:
    for i, (tr, te) in enumerate(tscv.split(np.zeros(n_samples))):
        ax.scatter(tr, [i] * len(tr), c="tab:blue", s=15, label="train" if i == 0 else "")
        ax.scatter(te, [i] * len(te), c="tab:red", s=15, label="test" if i == 0 else "")
    ax.set_ylabel("Fold")
    ax.set_title(title, fontweight="bold")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
axes[1].set_xlabel("Sample index (time)")
plt.tight_layout()
plt.show()

# Example use: evaluate a regressor on a synthetic time series
rng = np.random.RandomState(0)
t = np.arange(500)
X_ts = t.reshape(-1, 1)
y_ts = 0.01 * t + np.sin(t / 20) + rng.normal(0, 0.2, size=500)

scores_ts = cross_val_score(
    Ridge(alpha=1.0),
    X_ts,
    y_ts,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="r2",
)
print(f"TimeSeriesSplit(5) R^2 per fold: {scores_ts.round(4).tolist()}")
print(f"Mean: {scores_ts.mean():.4f}")

# 11. Bayesian Hyperparameter Search

Grid and random search are **uninformed**: they treat each trial as independent. Bayesian optimization does better by:

1. Fitting a **surrogate model** (usually a Gaussian Process or a tree-based model) to the (hyperparameters → CV score) points it has seen so far.
2. Picking the next hyperparameter configuration by maximizing an **acquisition function** (e.g. Expected Improvement) — balancing exploration (high-uncertainty regions) with exploitation (high-predicted-score regions).
3. Evaluating that configuration, updating the surrogate, repeating.

On expensive objectives (large models, long CV), Bayesian search typically finds a near-optimal configuration in **far fewer evaluations** than random search — often 5–10× faster.

Popular libraries:
* **`scikit-optimize`** (`skopt`): drop-in `BayesSearchCV` with the sklearn CV API. We use it here.
* **`optuna`**: modern, flexible, great for complex search spaces.
* **`hyperopt`**: older, Tree-Parzen Estimator-based.

## 11.1 `BayesSearchCV` — drop-in replacement for `RandomizedSearchCV`

Three things to know:
* `Real(low, high, prior='log-uniform')` — continuous log-scale dimension (for `alpha`, `C`, `gamma`…).
* `Integer(low, high)` — integer dimension.
* `Categorical([...])` — discrete choices.

The number of trials is controlled by `n_iter`.

In [ ]:
# Bayesian search on an ElasticNet: 2-D continuous space (alpha, l1_ratio)
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from skopt import BayesSearchCV
from skopt.space import Real

pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("enet", ElasticNet(max_iter=20000, random_state=42)),
    ]
)

search_spaces = {
    "enet__alpha": Real(1e-5, 1e2, prior="log-uniform"),
    "enet__l1_ratio": Real(0.0, 1.0, prior="uniform"),
}

bayes_search = BayesSearchCV(
    pipe,
    search_spaces=search_spaces,
    n_iter=30,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
)
bayes_search.fit(X_train_rs, y_train_r)

print(f"Best params: {dict(bayes_search.best_params_)}")
print(f"Best CV R^2: {bayes_search.best_score_:.4f}")

In [ ]:
# Visualize convergence: best CV score as a function of number of trials
scores_over_iters = np.maximum.accumulate(bayes_search.cv_results_["mean_test_score"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, len(scores_over_iters) + 1), scores_over_iters, "o-", markersize=5)
ax.set_xlabel("Number of trials")
ax.set_ylabel("Best CV R^2 so far")
ax.set_title("BayesSearchCV convergence", fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Bayesian search converges much faster than random search once the surrogate")
print("has enough data — the first ~10 trials look random, then it focuses in.")

## 11.2 Comparison: Random Search vs Bayesian Search (same budget)

In [ ]:
from scipy.stats import loguniform, uniform
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    pipe,
    param_distributions={
        "enet__alpha": loguniform(1e-5, 1e2),
        "enet__l1_ratio": uniform(0.0, 1.0),
    },
    n_iter=30,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
)
random_search.fit(X_train_rs, y_train_r)

# Cumulative best score over iterations — Bayesian vs Random
best_bayes = np.maximum.accumulate(bayes_search.cv_results_["mean_test_score"])
best_rand = np.maximum.accumulate(random_search.cv_results_["mean_test_score"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, len(best_bayes) + 1), best_bayes, "o-", label="BayesSearchCV")
ax.plot(np.arange(1, len(best_rand) + 1), best_rand, "s-", label="RandomizedSearchCV")
ax.set_xlabel("Number of trials")
ax.set_ylabel("Best CV R^2 so far")
ax.set_title("Bayesian vs Random search — convergence", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"BayesSearchCV best R^2:       {bayes_search.best_score_:.4f}")
print(f"RandomizedSearchCV best R^2:  {random_search.best_score_:.4f}")
print("\nOn a cheap objective like this the gap is often small —")
print("Bayesian search pays off most clearly when each CV evaluation is expensive.")

## 11.3 Bonus: Successive Halving (resource-efficient, not Bayesian)

Scikit-learn also ships with `HalvingGridSearchCV` and `HalvingRandomSearchCV`. The idea is different from Bayesian search: evaluate **all** configurations cheaply (small CV fold / small dataset), keep the top ones, run them on a bigger resource budget, repeat. Great when you have many configurations and a tunable resource (e.g. `n_estimators`).

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV

halving = HalvingRandomSearchCV(
    pipe,
    param_distributions={
        "enet__alpha": loguniform(1e-5, 1e2),
        "enet__l1_ratio": uniform(0.0, 1.0),
    },
    n_candidates=60,
    factor=3,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
)
halving.fit(X_train_rs, y_train_r)

print(f"HalvingRandomSearchCV best R^2:  {halving.best_score_:.4f}")
print(f"Best params: {halving.best_params_}")
print(f"\nUsed {halving.n_resources_[-1]} training samples in the final round (started at {halving.n_resources_[0]}).")

# Summary

| Chapter | Key sklearn/imblearn/skopt Classes |
|---------|------------------------------------|
| **1. Imputation** | `SimpleImputer`, `KNNImputer`, `IterativeImputer`, `MissingIndicator` |
| **2. Feature Engineering** | `MinMaxScaler`, `StandardScaler`, `RobustScaler`, `QuantileTransformer`, `PowerTransformer`, `KBinsDiscretizer`, `PolynomialFeatures`, `SplineTransformer`, `OneHotEncoder`, `OrdinalEncoder`, `TargetEncoder`, `LabelEncoder`, `ColumnTransformer` |
| **3. Regularization** | `Ridge`, `Lasso`, `ElasticNet`, `RidgeCV`, `LassoCV`, `ElasticNetCV`, `LogisticRegression(penalty=...)` |
| **4. Feature Selection** | `VarianceThreshold`, `SelectKBest`, `SelectPercentile`, `f_classif`, `chi2`, `mutual_info_classif`, `RFE`, `RFECV`, `SequentialFeatureSelector`, `SelectFromModel`, `permutation_importance` |
| **5. Rebalancing** | `RandomUnderSampler`, `ClusterCentroids`, `TomekLinks`, `EditedNearestNeighbours`, `RandomOverSampler`, `SMOTE`, `BorderlineSMOTE`, `SVMSMOTE`, `KMeansSMOTE`, `ADASYN`, `SMOTETomek`, `SMOTEENN` |
| **6. Ensemble** | `BaggingClassifier`, `StackingClassifier`, `VotingClassifier` |
| **7. Calibration** | `CalibratedClassifierCV`, `CalibrationDisplay`, `calibration_curve`, `brier_score_loss` |
| **8. Drift** | `scipy.stats.ks_2samp`, `scipy.stats.wasserstein_distance` |
| **9. Evaluation & Diagnostics** | `mean_squared_error`, `mean_absolute_error`, `mean_absolute_percentage_error`, `mean_squared_log_error`, `median_absolute_error`, `r2_score`, `matthews_corrcoef`, `fbeta_score`, `balanced_accuracy_score`, `roc_curve`, `roc_auc_score`, `precision_recall_curve`, `average_precision_score`, `log_loss`, `learning_curve`, `validation_curve` |
| **10. CV Variants** | `LeaveOneOut`, `LeavePOut`, `RepeatedKFold`, `RepeatedStratifiedKFold`, `TimeSeriesSplit`, nested CV (`cross_val_score(GridSearchCV(...))`) |
| **11. Bayesian Search** | `skopt.BayesSearchCV`, `sklearn.model_selection.HalvingRandomSearchCV`, `HalvingGridSearchCV` |
